# Phase 3 - Level 1 (v3, fixed key-crop share): geographic diversity - GOOGLE COLAB
Source = USA states only (Sentinel-2 medians, **not** AlphaEarth). Targets = ARG, FRA, CHN, AUS (never used for any decision). Model = the authors' **CropNet** (`x-ytong/CropNet` @ `bdcf63ec1704a412da73c534c579fe08c5d2fa9b`, imported unchanged), all 7 outputs trained.
Implements the written spec exactly: key-crop share fixed at every k, the rest natural; binary is an **evaluation view only**; paper rule from the authors' `util/label.py::remap_prediction`; early stopping on validation macro-F1 inside 50 epochs; two independent scorers; integrity gates vs skill warnings.

**Order (spec 9, enforced by the cells):** tests (Parts A + B) -> data check + **plan table (you approve)** -> 2-epoch smoke test -> **seed 0** sweep + gate report (you approve) -> seeds 1-4, each with its own gate report. Level 2 is a separate approval stage.
Approvals are the `APPROVED` flags in the config cell; `RUN_SEED` selects the seed. Results go to `outputs/phase3_level1_v3_fixedmix` (a folder from an earlier notebook is refused, never overwritten).

In [ ]:
# ======================================================================================
# [COLAB] ENVIRONMENT: mount Drive, paths (your existing files), libraries, GPU check.
# ======================================================================================
import os, sys, subprocess, importlib.util, glob, json, time
PLATFORM = 'colab'
assert importlib.util.find_spec('google.colab') is not None, 'This notebook is for Google Colab. Use the Kaggle notebook on Kaggle.'
from google.colab import drive
drive.mount('/content/drive')
_candidates = ['/content/drive/.shortcut-targets-by-id/1HTRsB7bxydWFYCZG8Wrj-8IQZJKj2Z8w/cropglobe_assignment',
               '/content/drive/MyDrive/cropglobe_assignment']
PROJECT_ROOT = next((p for p in _candidates if os.path.isdir(p)), None)
assert PROJECT_ROOT, f'project folder not found. Looked in: {_candidates}. Edit PROJECT_ROOT here.'
DATA_DIR   = os.path.join(PROJECT_ROOT, 'CropGlobe')                     # contains for_CropNet/
RAW_USA_DIR = os.path.join(PROJECT_ROOT, 'CropGlobe_raw', 'Sentinel-2', 'USA')
REPO_DIR   = os.path.join(PROJECT_ROOT, 'CropNet')
NAMESPACE  = 'phase3_level1_v3_fixedmix'
OUT_DIR    = os.path.join(PROJECT_ROOT, 'outputs', NAMESPACE)            # NEW folder: the legacy results (outputs/phase3 + backup) are never touched
os.makedirs(OUT_DIR, exist_ok=True)
PLATFORM_TIME_LIMIT_MIN = None                                           # no artificial limit; re-run the run cell after a disconnect to resume
EXTRA_META_PATHS = []
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub', 'geopandas', 'shapely', 'pyproj', 'scikit-learn', 'scipy', 'tqdm'], check=True)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE -> Runtime > Change runtime type > GPU')
assert torch.cuda.is_available(), 'enable a GPU runtime'
print('project:', PROJECT_ROOT, '\noutputs:', OUT_DIR)

## Configuration (locked design, approvals)

In [ ]:
# ======================================================================================
# CONFIGURATION  (pure python; nothing here touches the network or the GPU)
# Level 1 - geographic diversity with the KEY-CROP share held fixed.  Implements the written spec EXACTLY.
# ======================================================================================
import os, sys, json, math, time, glob, re, csv, zlib, random, itertools, hashlib, datetime, pickle, warnings, traceback, subprocess, tempfile, io, contextlib
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
from scipy import stats as scipy_stats
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_recall_fscore_support

EXPERIMENT_VERSION = 'phase3_level1_v3_fixedmix'
SPEC_TAG = 'level1_fixedmix_spec_v1'          # written into the manifest; a folder without it is NOT ours -> the run stops instead of mixing

# ---------------- design (spec section 4) ----------------
BUDGETS      = [1000, 2000, 3000, 4000]       # TOTAL labelled samples per run (train + validation); never shrunk
K_LIST       = [1, 2, 4, 8]
SEEDS        = [0, 1, 2, 3, 4]
AUG          = 'off'                          # Level 1 = augmentation OFF only (spec 2).  ON needs a step-matched control + approval.
MAX_COMBOS   = 10                             # k>=2: up to 10 random combinations; k=1 is a census of EVERY pool state
MIN_COMBOS   = 5                              # a (target,budget,k) cell with fewer combinations is recorded as infeasible
MIN_KPOINTS  = 3                              # a (target,budget) with fewer feasible k-points is infeasible as a whole
MIN_STATE_ROWS = 1500                         # source pool = USA states with >= 1500 (de-duplicated) rows
VAL_FRACTION = 0.10                           # validation = 10% of the budget, stratified by crop, deterministic
TARGETS_ORDER = ['ARG', 'FRA', 'CHN', 'AUS']
TARGETS_TO_RUN = ['ARG', 'FRA', 'CHN', 'AUS']
TARGET_ROLE  = {'ARG': 'main', 'FRA': 'main', 'CHN': 'secondary', 'AUS': 'appendix'}
# KEY-CROP share (fixed, identical at every k); everything else is sampled NATURALLY from the state's remaining rows.
KEY_SHARES   = {'ARG': {'soybeans': 0.20}, 'CHN': {'rice': 0.20}, 'FRA': {'corn': 0.15, 'soybeans': 0.15, 'wheat': 0.15}, 'AUS': None}
K_MAX        = {'ARG': 8, 'FRA': 8, 'CHN': 4, 'AUS': 2}
FRA_K8_MIN_POOL = 10                          # FRA k=8 only if the pool has >= 10 states
AUS_STATES   = ['LA', 'FL']                   # AUS = appendix census of {LA, FL}, k <= 2, no sweep (exempt from MIN_COMBOS / MIN_KPOINTS)
AUS_SPEC_CONFIRMED = False                    # !! the spec gives NO key-crop share for AUS.  AUS stays OUT of the run until you decide (see plan table).
AUS_KEY_SHARES_IF_CONFIRMED = None            # e.g. {'sugarcane': 0.20}; None = fully natural sampling.  Set deliberately.

# ---------------- training (spec 2; do not change) ----------------
NUM_CLASSES  = 7                              # CropNet(data_channel=1, classes=7): ALL 7 outputs trained, never a binary head
LR, WEIGHT_DECAY, BATCH_SIZE = 1e-4, 0.0, 256 # Adam, no scheduler
MAX_EPOCHS, PATIENCE, MIN_EPOCHS, MIN_DELTA = 50, 15, 20, 0.0     # early stopping INSIDE the 50 epochs, on VALIDATION macro-F1, strictly greater
UNDERTRAINED_EPOCH = 45                       # best_epoch (1-based) >= 45  -> flag 'possibly_undertrained'
RESTORE_TOL  = 1e-6                           # restored model must reproduce its best validation score
EXPECTED_CROPNET_COMMIT = 'bdcf63ec1704a412da73c534c579fe08c5d2fa9b'

# ---------------- label space ----------------
CROP_LABELS = {0: 'corn', 1: 'soybeans', 2: 'rice', 3: 'wheat', 4: 'sugarcane', 5: 'cotton', 6: 'other'}
CROP_ID     = {v: k for k, v in CROP_LABELS.items()}
CROP_ORDER  = [CROP_LABELS[i] for i in range(7)]
RAW_TO_CROP = {'corn': 'corn', 'soybeans': 'soybeans', 'rice': 'rice', 'springwheat': 'wheat', 'winterwheat': 'wheat',
               'wheat': 'wheat', 'sugarcane': 'sugarcane', 'cotton': 'cotton', 'other': 'other'}

# paper (arXiv 2509.03497 v3) Table 2 -- raw class counts.  Raw counts and label sets are ASSERTED against this (stop on mismatch).
TABLE2 = {
    'ARG': dict(corn=1630,  soybeans=1808,  rice=1319,  wheat=0,     sugarcane=1253, cotton=1443,  other=2425),
    'AUS': dict(corn=0,     soybeans=0,     rice=0,     wheat=0,     sugarcane=5098, cotton=4825,  other=8938),
    'CHN': dict(corn=1344,  soybeans=1200,  rice=1156,  wheat=0,     sugarcane=0,    cotton=0,     other=1502),
    'FRA': dict(corn=24902, soybeans=21002, rice=2294,  wheat=27686, sugarcane=0,    cotton=0,     other=32715),
    'USA': dict(corn=22224, soybeans=17093, rice=17613, wheat=23226, sugarcane=9897, cotton=14799, other=28427),
}
TABLE2_TOTAL = {'ARG': 9878, 'AUS': 18861, 'CHN': 5202, 'FRA': 108599, 'USA': 133279}
assert all(sum(TABLE2[c].values()) == TABLE2_TOTAL[c] for c in TABLE2), 'Table 2 transcription error'
DEFINED_CLASSES = {t: [c for c in CROP_ORDER if TABLE2[t][c] > 0] for t in TABLE2}     # label set per country = classes with count > 0 in Table 2

# ---------------- scoring (spec 3) ----------------
PRIMARY_VIEW = {'ARG': ('binary', 'soybeans'), 'CHN': ('binary', 'rice'), 'AUS': ('paper', None), 'FRA': ('paper', None)}
BINARY_TARGETS = {t: v[1] for t, v in PRIMARY_VIEW.items() if v[0] == 'binary'}
COVERAGE_MIN_TRAIN = 30                        # 'coverage-conditional' macro-F1: classes with >= 30 TRAINING samples (reported next to, never instead of, the main score)
SCORER_TOL = 1e-12                             # sklearn scorer vs from-scratch numpy scorer
# paper Table 3, CropNet_AUGM, FULL USA source, mF1 (%).  CONTEXT ONLY: NOT comparable to this design (budgets of 1-4k, augmentation off). No flag is derived from it.
TABLE3_REF_MF1 = {'ARG': 75.63, 'AUS': 88.38, 'FRA': 90.00}      # (the paper has no USA->CHN experiment)

# ---------------- gates (spec 6) ----------------
VAL_F1_WARN = 0.60                             # SKILL outcome: warn only, never abort
MAX_CONSEC_INTEGRITY = 5                       # abort only after this many CONSECUTIVE integrity failures
INTEGRITY_CHECKS = ['I1_true_classes_ge2', 'I2_predictions_valid', 'I3_split_exact', 'I4_restore_matches_best', 'I5_finite', 'I6_scorers_agree']

# ---------------- approvals (spec 9): nothing trains until you flip these, in order ----------------
APPROVED = {'plan': False, 'seed0_gate_report': False, 'seed1_gate_report': False, 'seed2_gate_report': False, 'seed3_gate_report': False}
RUN_SEED = 0                                   # which seed the run cell executes this time (seed 0 first; seed s>0 needs APPROVED[f'seed{s-1}_gate_report'])

LOCKED = ['BUDGETS', 'K_LIST', 'SEEDS', 'AUG', 'MAX_COMBOS', 'MIN_COMBOS', 'MIN_KPOINTS', 'MIN_STATE_ROWS', 'VAL_FRACTION', 'KEY_SHARES', 'K_MAX', 'FRA_K8_MIN_POOL',
          'AUS_STATES', 'NUM_CLASSES', 'LR', 'WEIGHT_DECAY', 'BATCH_SIZE', 'MAX_EPOCHS', 'PATIENCE', 'MIN_EPOCHS', 'MIN_DELTA', 'PRIMARY_VIEW', 'COVERAGE_MIN_TRAIN',
          'EXPECTED_CROPNET_COMMIT']
IN_NOTEBOOK = 'ipykernel' in sys.modules
print('config OK |', EXPERIMENT_VERSION, '| spec', SPEC_TAG)

## Core logic: scoring (two scorers), gates, sampling, plan, statistics, atomic outputs

In [ ]:
# ======================================================================================
# CORE LOGIC (pure numpy / pandas / sklearn -- no torch, no network).  Unit-tested in the next cell.
# ======================================================================================
class TooFewClasses(Exception): pass
class InfeasibleSplit(Exception): pass
class IntegrityFailure(Exception): pass

def _check(cond, msg):
    if not cond:                      # cannot be disabled with python -O
        raise AssertionError(msg)

def stable_seed(*parts):
    return zlib.crc32('|'.join(str(p) for p in parts).encode()) & 0x7fffffff

def atomic_write(path, writer, mode='wb'):
    # write to a temp file in the same folder, fsync, then os.replace: a reader never sees a half-written file
    d = os.path.dirname(path); os.makedirs(d, exist_ok=True)
    fd, tmp = tempfile.mkstemp(dir=d, prefix='.tmp_', suffix=os.path.basename(path))
    try:
        with os.fdopen(fd, mode) as f:
            writer(f); f.flush(); os.fsync(f.fileno())
        os.replace(tmp, path)
    except BaseException:
        if os.path.exists(tmp): os.remove(tmp)
        raise

# ------------------------------------------------------------------------------ SCORING (spec 3)
def binary_collapse(y_true, y_pred, positive_name):
    # EVALUATION VIEW ONLY.  Every non-positive class, true OR predicted, is 'other'.  Nothing is dropped, nothing is set to -1.
    pid = CROP_ID[positive_name]
    return (np.asarray(y_true) == pid).astype(int), (np.asarray(y_pred) == pid).astype(int)

def remap_paper_rule(y_pred, target):
    # PAPER RULE = the authors' released util/label.py::remap_prediction (applied to PREDICTIONS only):
    #   AUS keep {sugarcane, cotton} | CHN keep {corn, soybeans, rice} | ARG only wheat -> other | FRA no remap (a wrong crop stays an error)
    y = np.asarray(y_pred).astype(int); o = CROP_ID['other']
    if target == 'AUS': return np.where(np.isin(y, [CROP_ID['sugarcane'], CROP_ID['cotton']]), y, o)
    if target == 'CHN': return np.where(np.isin(y, [CROP_ID['corn'], CROP_ID['soybeans'], CROP_ID['rice']]), y, o)
    if target == 'ARG': return np.where(y == CROP_ID['wheat'], o, y)
    if target == 'FRA': return y.copy()
    raise ValueError(f'unknown target {target}')

def _np_prf(yt, yp, labels):
    # scorer #2: from scratch, numpy only (no sklearn)
    P, R, F = [], [], []
    for c in labels:
        tp = float(np.sum((yt == c) & (yp == c))); fp = float(np.sum((yt != c) & (yp == c))); fn = float(np.sum((yt == c) & (yp != c)))
        P.append(tp / (tp + fp) if tp + fp > 0 else 0.0); R.append(tp / (tp + fn) if tp + fn > 0 else 0.0)
        d = 2 * tp + fp + fn; F.append(2 * tp / d if d > 0 else 0.0)
    return np.array(P), np.array(R), np.array(F)

def _sk_prf(yt, yp, labels):
    p, r, f, _ = precision_recall_fscore_support(yt, yp, labels=list(labels), zero_division=0)
    return p, r, f

def _expand(vals, labels, n=7):
    out = np.full(n, np.nan); out[np.asarray(labels, dtype=int)] = vals; return out

def macro_paper_rule_sklearn(y_true, y_pred_remapped):
    return float(f1_score(y_true, y_pred_remapped, average='macro', labels=np.unique(y_true), zero_division=0))

def score_run(target, y_true, y_pred, train_counts):
    # y_true / y_pred : integer ids 0..6 of the FULL target set.  train_counts : crop counts (length 7) of the TRAIN split.
    yt = np.asarray(y_true).astype(int); yp = np.asarray(y_pred).astype(int); tc = np.asarray(train_counts).astype(int)
    _check(yt.ndim == 1 and yt.shape == yp.shape and len(yt) > 0, 'y_true / y_pred empty or length mismatch')
    _check(yt.min() >= 0 and yt.max() <= 6 and yp.min() >= 0 and yp.max() <= 6, 'class ids outside 0..6')
    present = np.unique(yt); O = CROP_ID['other']
    if len(present) < 2:
        raise TooFewClasses(f'{target}: true labels contain {len(present)} class(es) -- need >= 2; run stopped')
    yr = remap_paper_rule(yp, target)
    # ---- paper rule: macro-F1 over the classes present in y_true, OA over ALL samples
    p_s, r_s, f_s = _sk_prf(yt, yr, present); p_n, r_n, f_n = _np_prf(yt, yr, present)
    paper_f1, paper_oa = macro_paper_rule_sklearn(yt, yr), float(accuracy_score(yt, yr))
    paper_f1_n, paper_oa_n = float(f_n.mean()), float((yt == yr).mean())
    diffs = [abs(paper_f1 - paper_f1_n), abs(paper_oa - paper_oa_n), *np.abs(f_s - f_n), *np.abs(p_s - p_n), *np.abs(r_s - r_n)]
    # ---- baselines in the paper-rule view (all-'other'; majority class of the true labels), scored with the same rule
    maj = int(np.bincount(yt, minlength=7).argmax())
    base = dict(paper_all_other=float(_np_prf(yt, remap_paper_rule(np.full_like(yt, O), target), present)[2].mean()),
                paper_majority=float(_np_prf(yt, remap_paper_rule(np.full_like(yt, maj), target), present)[2].mean()))
    # ---- coverage-conditional (labelled; never replaces the main score): classes with >= 30 TRAINING samples
    cov_idx = [i for i, c in enumerate(present) if tc[c] >= COVERAGE_MIN_TRAIN]
    cov_paper = float(np.mean(f_s[cov_idx])) if cov_idx else float('nan')
    absent_train = [CROP_LABELS[int(c)] for c in present if tc[c] == 0]            # stay IN the score at their F1; only flagged
    res = dict(n=int(len(yt)), n_true_classes=int(len(present)), present=[CROP_LABELS[int(c)] for c in present],
               paper_f1=paper_f1, paper_oa=paper_oa, paper_prec=_expand(p_s, present), paper_rec=_expand(r_s, present), paper_f1_cls=_expand(f_s, present),
               cov_cond_paper_f1=cov_paper, cov_cond_paper_classes=[CROP_LABELS[int(present[i])] for i in cov_idx], absent_train_classes=absent_train,
               yp_remapped=yr, cm7=confusion_matrix(yt, yp, labels=list(range(7))), cm_paper=confusion_matrix(yt, yr, labels=list(range(7))),
               bin_f1=float('nan'), bin_oa=float('nan'), cm_binary=None, cov_cond_binary_f1=float('nan'), bin_prec=None, bin_rec=None, bin_f1_cls=None,
               baseline_all_other_primary=base['paper_all_other'], baseline_majority_primary=base['paper_majority'], baseline_all_other_paper=base['paper_all_other'],
               baseline_majority_paper=base['paper_majority'], primary_view='paper_rule', positive=None)
    if target in BINARY_TARGETS:
        pos = BINARY_TARGETS[target]; pid = CROP_ID[pos]; bt, bp = binary_collapse(yt, yp, pos); lab = [0, 1]
        pb_s, rb_s, fb_s = _sk_prf(bt, bp, lab); pb_n, rb_n, fb_n = _np_prf(bt, bp, lab)
        bin_f1 = float(f1_score(bt, bp, average='macro', labels=lab, zero_division=0)); bin_oa = float(accuracy_score(bt, bp))
        diffs += [abs(bin_f1 - float(fb_n.mean())), abs(bin_oa - float((bt == bp).mean())), *np.abs(fb_s - fb_n), *np.abs(pb_s - pb_n), *np.abs(rb_s - rb_n)]
        n_pos_tr, n_oth_tr = int(tc[pid]), int(tc.sum() - tc[pid]); cov_b = [i for i, n in enumerate([n_oth_tr, n_pos_tr]) if n >= COVERAGE_MIN_TRAIN]   # index 0 = other, 1 = positive
        maj_b = int(np.bincount(bt, minlength=2).argmax())
        res.update(bin_f1=bin_f1, bin_oa=bin_oa, cm_binary=confusion_matrix(bt, bp, labels=lab), bin_prec=pb_s, bin_rec=rb_s, bin_f1_cls=fb_s,
                   cov_cond_binary_f1=float(np.mean(fb_s[cov_b])) if cov_b else float('nan'), primary_view=f'binary_{pos}', positive=pos,
                   baseline_all_other_primary=float(_np_prf(bt, np.zeros_like(bt), lab)[2].mean()), baseline_majority_primary=float(_np_prf(bt, np.full_like(bt, maj_b), lab)[2].mean()),
                   binary_pred_unique=int(len(np.unique(bp))))
        if n_pos_tr == 0: res['absent_train_classes'] = sorted(set(res['absent_train_classes']) | {pos})
    res['primary_f1'] = res['bin_f1'] if target in BINARY_TARGETS else res['paper_f1']
    res['primary_oa'] = res['bin_oa'] if target in BINARY_TARGETS else res['paper_oa']
    res['baseline_best_primary'] = max(res['baseline_all_other_primary'], res['baseline_majority_primary'])
    res['scorer_maxdiff'] = float(max(diffs))
    res['primary_pred_unique'] = res.get('binary_pred_unique', int(len(np.unique(yr))))
    return res

# ------------------------------------------------------------------------------ GATES (spec 6)
def evaluate_integrity(sc, preds, split_chk, info):
    # INTEGRITY failures (abort after 5 consecutive).  Skill outcomes are NOT here.
    p = np.asarray(preds); n_test = sc['n']
    finite_vals = [sc['primary_f1'], sc['paper_f1'], sc['paper_oa'], sc['baseline_all_other_primary'], sc['baseline_majority_primary'], info.get('val_f1', np.nan), info.get('restore_delta', np.nan)]
    if sc['cm_binary'] is not None: finite_vals += [sc['bin_f1'], sc['bin_oa']]
    I = {'I1_true_classes_ge2': bool(sc['n_true_classes'] >= 2),
         'I2_predictions_valid': bool(len(p) == n_test and p.min() >= 0 and p.max() <= 6 and int(sc['cm7'].sum()) == n_test and int(sc['cm_paper'].sum()) == n_test
                                       and (sc['cm_binary'] is None or int(sc['cm_binary'].sum()) == n_test)),
         'I3_split_exact': bool(split_chk['ok']),
         'I4_restore_matches_best': bool(info.get('restore_delta', np.inf) <= RESTORE_TOL),
         'I5_finite': bool(all(np.isfinite(finite_vals))),
         'I6_scorers_agree': bool(sc['scorer_maxdiff'] <= SCORER_TOL)}
    return I, all(I.values())

def skill_flags(sc, info):
    # SKILL outcomes: recorded and warned, NEVER an abort
    f = []
    if info.get('val_f1', 1.0) < VAL_F1_WARN: f.append(f'WARN_VAL_F1_LT_{VAL_F1_WARN:.2f}')
    if not (sc['primary_f1'] > sc['baseline_best_primary']): f.append('WARN_NOT_BETTER_THAN_BASELINE')
    if sc['primary_pred_unique'] < 2: f.append('WARN_COLLAPSED_PREDICTIONS')
    if info.get('best_epoch', 0) >= UNDERTRAINED_EPOCH: f.append('possibly_undertrained')
    if sc['absent_train_classes']: f.append('FLAG_ABSENT_TRAIN_CLASS:' + '+'.join(sc['absent_train_classes']))
    return f

# ------------------------------------------------------------------------------ EARLY STOPPING (pure; the torch loop uses it)
class EarlyStopper:
    # validation macro-F1; strictly greater than best + MIN_DELTA counts as an improvement; never more than max_epochs
    def __init__(self, max_epochs=None, patience=None, min_epochs=None, min_delta=None):
        self.max_epochs = MAX_EPOCHS if max_epochs is None else max_epochs; self.patience = PATIENCE if patience is None else patience
        self.min_epochs = MIN_EPOCHS if min_epochs is None else min_epochs; self.min_delta = MIN_DELTA if min_delta is None else min_delta
        _check(self.max_epochs <= MAX_EPOCHS, f'max_epochs {self.max_epochs} would exceed the hard cap {MAX_EPOCHS}')
        self.best, self.best_epoch, self.bad, self.epoch, self.reason = -np.inf, 0, 0, 0, None
    def step(self, score):
        self.epoch += 1; improved = score > self.best + self.min_delta
        if improved: self.best, self.best_epoch, self.bad = score, self.epoch, 0
        else: self.bad += 1
        if self.epoch >= self.min_epochs and self.bad >= self.patience: self.reason = 'early_stop_patience'
        elif self.epoch >= self.max_epochs: self.reason = 'max_epochs'
        return improved, self.reason is not None

def balanced_class_weights(y, n_classes=7):
    # == sklearn compute_class_weight('balanced') over the classes present; classes with ZERO samples get weight 0 (no division by zero)
    y = np.asarray(y).astype(int); counts = np.bincount(y, minlength=n_classes).astype(float)
    present = counts > 0; w = np.zeros(n_classes)
    w[present] = len(y) / (present.sum() * counts[present])
    return w

# ------------------------------------------------------------------------------ DESIGN (spec 4): key-crop share fixed, the rest natural
def effective_key_shares(target):
    if target == 'AUS': return dict(AUS_KEY_SHARES_IF_CONFIRMED or {})
    return dict(KEY_SHARES[target])

def key_counts(target, B):
    out = {}
    for c, sh in effective_key_shares(target).items():
        n = sh * B; _check(abs(n - round(n)) < 1e-9, f'{target} B={B}: key share {sh} gives non-integer count {n}'); out[c] = int(round(n))
    return out

def rest_count(target, B): return B - sum(key_counts(target, B).values())

def eligible_states(target, B, cap):
    # pool = states that (i) have >= MIN_STATE_ROWS rows and (ii) can supply the WHOLE budget ALONE with the key share:
    #        key crop(s) >= their fixed counts AND all remaining (non-key) rows >= the natural remainder.   Same pool for every k.
    kc = key_counts(target, B); rest = B - sum(kc.values())
    ok = cap['total'].values >= max(MIN_STATE_ROWS, B)
    for c, n in kc.items(): ok &= cap[c].values >= n
    nonkey = cap['total'].values - sum((cap[c].values for c in kc), 0)
    ok &= nonkey >= rest
    pool = sorted(cap.index[ok].tolist())
    if target == 'AUS': pool = [s for s in pool if s in AUS_STATES]
    return pool

def make_combos(pool, k, target, B, max_combos=None):
    max_combos = MAX_COMBOS if max_combos is None else max_combos
    pool = sorted(pool)
    if k > len(pool): return []
    if k == 1: return [(s,) for s in pool]                       # CENSUS: every pool state
    total = math.comb(len(pool), k)
    if total <= max_combos: return list(itertools.combinations(pool, k))
    rng = np.random.default_rng(stable_seed('combo', target, B, k))   # deterministic seed from (target, budget, k)
    seen = set()
    while len(seen) < max_combos:
        seen.add(tuple(sorted(rng.choice(len(pool), size=k, replace=False).tolist())))
    return [tuple(pool[i] for i in idx) for idx in sorted(seen)]

def build_plan(cap):
    cells, grid = [], []
    for t in TARGETS_TO_RUN:
        pending = (t == 'AUS' and not AUS_SPEC_CONFIRMED)
        for B in BUDGETS:
            pool = eligible_states(t, B, cap); P = len(pool); kc = key_counts(t, B); rest = B - sum(kc.values()); rows = []
            for k in K_LIST:
                note, combos = '', []
                if k > K_MAX[t]: note = f'k > K_MAX({K_MAX[t]})'
                elif t == 'FRA' and k == 8 and P < FRA_K8_MIN_POOL: note = f'FRA k=8 needs pool >= {FRA_K8_MIN_POOL} (pool={P})'
                elif k > P: note = 'pool smaller than k'
                else: combos = make_combos(pool, k, t, B)
                if t != 'AUS' and combos and len(combos) < MIN_COMBOS: note, combos = f'{len(combos)} combinations < {MIN_COMBOS}', []
                rows.append(dict(k=k, combos=combos, note=note))
            if t != 'AUS' and sum(1 for r in rows if r['combos']) < MIN_KPOINTS:
                for r in rows:
                    if r['combos']: r['note'], r['combos'] = f'fewer than {MIN_KPOINTS} feasible k-points for this budget', []
            for r in rows:
                k = r['k']; ntot = math.comb(P, k) if k <= P else 0
                grid.append(dict(target=t, role=TARGET_ROLE[t], budget=B, k=k, pool_size=P, n_total_combos=ntot, n_combos=len(r['combos']), feasible=bool(r['combos']),
                                 note=r['note'] + (' [AUS PENDING: key share not specified]' if pending and r['combos'] else ''), key_counts=json.dumps(kc), rest=rest,
                                 pool=','.join(pool) if P <= 20 else f'{P} states'))
                for ci, c in enumerate(r['combos']):
                    cells.append(dict(target=t, budget=B, k=k, combo_id=ci, states=tuple(c), pool_size=P, n_total_combos=ntot, pending=pending))
    return cells, pd.DataFrame(grid)

def build_pools(usa_state, usa_crop):
    df = pd.DataFrame({'s': np.asarray(usa_state), 'c': np.asarray(usa_crop), 'i': np.arange(len(usa_state))})
    return {(s, c): g['i'].values.astype(np.int64) for (s, c), g in df.groupby(['s', 'c'])}

def make_cap(usa_state, usa_crop):
    cap = pd.crosstab(np.asarray(usa_state), np.asarray(usa_crop)).reindex(columns=CROP_ORDER, fill_value=0)
    cap['total'] = cap[CROP_ORDER].sum(axis=1); return cap

def allocate_val(counts, total, frac):
    # largest-remainder allocation of EXACTLY `total` validation rows across crops (stratified); every crop with >= 2 rows keeps >= 1 row in train
    ideal = {c: frac * n for c, n in counts.items()}
    v = {c: min(int(math.floor(ideal[c] + 1e-9)), max(n - 1, 0)) for c, n in counts.items()}
    for strict in (True, False):
        order = sorted(counts, key=lambda c: (-(ideal[c] - v[c]), CROP_ID[c]))
        while total - sum(v.values()) > 0:
            moved = False
            for c in order:
                if total - sum(v.values()) == 0: break
                if v[c] < counts[c] - 1 and (not strict or v[c] < math.ceil(ideal[c] - 1e-9)): v[c] += 1; moved = True
            if not moved: break
        if sum(v.values()) == total: return v
    raise InfeasibleSplit(f'cannot allocate {total} validation rows over {counts}')

def draw_split(target, B, states, seed, pools, usa_crop):
    # EXACTLY B rows.  Key crop(s): fixed counts (share x B), split equally over the states.  The remainder: natural draw from each state's non-key rows (equal count per state).
    states = tuple(sorted(states)); k = len(states); kc = key_counts(target, B); rest = B - sum(kc.values())
    rng = np.random.default_rng(stable_seed('draw', target, B, '-'.join(states), seed)); chosen = []
    def split_counts(n):
        base, rem = divmod(n, k); extra = set(rng.choice(k, size=rem, replace=False).tolist()) if rem else set()
        return [base + (1 if i in extra else 0) for i in range(k)]
    for c in sorted(kc, key=lambda x: CROP_ID[x]):
        cnt = split_counts(kc[c])
        for i, s in enumerate(states):
            pool = pools.get((s, c), np.zeros(0, dtype=np.int64))
            if len(pool) < cnt[i]: raise InfeasibleSplit(f'{target} B={B}: state {s} has {len(pool)} {c} rows, needs {cnt[i]}')
            chosen.append(rng.choice(pool, size=cnt[i], replace=False))
    cnt = split_counts(rest)
    for i, s in enumerate(states):
        parts = [pools[(s, c)] for c in CROP_ORDER if c not in kc and (s, c) in pools]; pool = np.concatenate(parts) if parts else np.zeros(0, dtype=np.int64)
        if len(pool) < cnt[i]: raise InfeasibleSplit(f'{target} B={B}: state {s} has {len(pool)} non-key rows, needs {cnt[i]}')
        chosen.append(rng.choice(pool, size=cnt[i], replace=False))
    allp = np.concatenate(chosen); crops = np.asarray(usa_crop)[allp]
    nval = int(round(VAL_FRACTION * B)); _check(abs(VAL_FRACTION * B - nval) < 1e-9, 'validation size must be an integer')
    counts = {c: int((crops == c).sum()) for c in CROP_ORDER if (crops == c).any()}; vq = allocate_val(counts, nval, VAL_FRACTION); tr, va = [], []
    for c in counts:
        pos = allp[crops == c]; perm = np.random.default_rng(stable_seed('val', target, B, '-'.join(states), seed, c)).permutation(len(pos))
        va.append(pos[perm[:vq[c]]]); tr.append(pos[perm[vq[c]:]])
    return dict(train_pos=np.concatenate(tr), val_pos=np.concatenate(va), key_counts=kc, rest=rest, states=states, val_quota=vq)

def verify_split(split, target, B, states, usa_state, usa_crop):
    msgs = []; tr, va = split['train_pos'], split['val_pos']; allp = np.concatenate([tr, va]); kc = key_counts(target, B); k = len(states)
    st, cr = np.asarray(usa_state), np.asarray(usa_crop)
    def need(c, m):
        if not c: msgs.append(m)
    nval = int(round(VAL_FRACTION * B))
    need(len(allp) == B, f'train+val={len(allp)} != budget {B}')
    need(len(va) == nval and len(tr) == B - nval, f'split sizes train={len(tr)} val={len(va)} != {B - nval}/{nval}')
    need(len(np.unique(allp)) == len(allp), 'duplicate rows inside the drawn sample (unique-id check)')
    need(len(np.intersect1d(tr, va)) == 0, 'train/validation overlap')
    need(allp.min() >= 0 and allp.max() < len(st), 'row id outside the USA table')
    need(set(st[allp].tolist()) <= set(states), 'a drawn row comes from a state outside the combination')
    for c, n in kc.items():
        need(int((cr[allp] == c).sum()) == n, f'key crop {c}: {int((cr[allp] == c).sum())} drawn vs fixed {n}')
        counts = [int(((cr[allp] == c) & (st[allp] == s)).sum()) for s in states]; need(max(counts) - min(counts) <= 1, f'key crop {c}: states not equally represented {counts}')
    nonkey = ~np.isin(cr[allp], list(kc)); rest_counts = [int((nonkey & (st[allp] == s)).sum()) for s in states]
    need(int(nonkey.sum()) == B - sum(kc.values()), 'non-key count != natural remainder'); need(max(rest_counts) - min(rest_counts) <= 1, f'non-key rows not equally split over states {rest_counts}')
    for c in set(cr[allp].tolist()):
        n_c = int((cr[allp] == c).sum()); v_c = int((cr[va] == c).sum())
        need(abs(v_c - VAL_FRACTION * n_c) <= 1.0 + 1e-9, f'validation not stratified for {c}: {v_c} of {n_c}')
    return dict(ok=not msgs, msgs=msgs)

# ------------------------------------------------------------------------------ RESULTS TABLE (atomic; the metrics file is written LAST)
def make_run_id(cell, seed):
    return f"{cell['target']}|B{cell['budget']}|k{cell['k']}|{'-'.join(cell['states'])}|aug-{AUG}|s{seed}"

COLUMNS = (['run_id', 'status', 'target', 'role', 'budget', 'k', 'combo_id', 'states', 'pool_size', 'n_total_combos', 'seed', 'n_train', 'n_val', 'epochs_run', 'best_epoch', 'stop_reason',
            'val_f1', 'restore_delta', 'primary_view', 'primary_f1', 'primary_oa', 'paper_f1', 'paper_oa', 'binary_f1', 'binary_oa',
            'baseline_all_other_primary', 'baseline_majority_primary', 'baseline_all_other_paper', 'baseline_majority_paper', 'cov_cond_primary_f1', 'cov_cond_paper_f1',
            'absent_train_classes', 'scorer_maxdiff', 'test_n', 'key_n']
           + [f'train_{c}' for c in CROP_ORDER] + [f'val_{c}' for c in CROP_ORDER] + [f'true_{c}' for c in CROP_ORDER] + [f'pred_{c}' for c in CROP_ORDER]
           + [f'f1_{c}' for c in CROP_ORDER] + INTEGRITY_CHECKS + ['integrity_pass', 'skill_flags', 'seconds', 'error'])

def load_results(path):
    if not os.path.exists(path): return pd.DataFrame(columns=COLUMNS)
    return pd.read_csv(path)

def append_row(path, row):
    df = load_results(path); df = pd.concat([df, pd.DataFrame([{c: row.get(c, '') for c in COLUMNS}])], ignore_index=True)
    atomic_write(path, lambda f: df.to_csv(f, index=False), mode='w')

def build_row(rid, cell, seed, split, sc, I, ok, skills, info, preds, y_true, seconds, usa_crop):
    tr, va = split['train_pos'], split['val_pos']; row = {c: '' for c in COLUMNS}; cr = np.asarray(usa_crop)
    row.update(run_id=rid, status='ok' if ok else 'integrity_fail', target=cell['target'], role=TARGET_ROLE[cell['target']], budget=cell['budget'], k=cell['k'], combo_id=cell['combo_id'],
               states='-'.join(cell['states']), pool_size=cell.get('pool_size', ''), n_total_combos=cell.get('n_total_combos', ''), seed=seed, n_train=len(tr), n_val=len(va),
               epochs_run=info.get('epochs_run', ''), best_epoch=info.get('best_epoch', ''), stop_reason=info.get('stop_reason', ''), val_f1=info['val_f1'], restore_delta=info.get('restore_delta', ''),
               primary_view=sc['primary_view'], primary_f1=sc['primary_f1'], primary_oa=sc['primary_oa'], paper_f1=sc['paper_f1'], paper_oa=sc['paper_oa'], binary_f1=sc['bin_f1'], binary_oa=sc['bin_oa'],
               baseline_all_other_primary=sc['baseline_all_other_primary'], baseline_majority_primary=sc['baseline_majority_primary'],
               baseline_all_other_paper=sc['baseline_all_other_paper'], baseline_majority_paper=sc['baseline_majority_paper'],
               cov_cond_primary_f1=(sc['cov_cond_binary_f1'] if cell['target'] in BINARY_TARGETS else sc['cov_cond_paper_f1']), cov_cond_paper_f1=sc['cov_cond_paper_f1'],
               absent_train_classes='+'.join(sc['absent_train_classes']), scorer_maxdiff=sc['scorer_maxdiff'], test_n=sc['n'], key_n=sum(split['key_counts'].values()),
               integrity_pass=ok, skill_flags='|'.join(skills), seconds=round(seconds, 2), error='')
    for c in CROP_ORDER:
        row[f'train_{c}'] = int((cr[tr] == c).sum()); row[f'val_{c}'] = int((cr[va] == c).sum())
        row[f'true_{c}'] = int((np.asarray(y_true) == CROP_ID[c]).sum()); row[f'pred_{c}'] = int((np.asarray(preds) == CROP_ID[c]).sum())
        row[f'f1_{c}'] = sc['paper_f1_cls'][CROP_ID[c]]
    row.update(I); return row

# ------------------------------------------------------------------------------ RUN ONE JOB + RUN LOOP
#   FIT_FN(Xtr, ytr, Xva, yva, seed, epochs_override) -> dict(model=..., info=dict(val_f1, best_epoch, epochs_run, stop_reason, restore_delta, history))
#   PREDICT_FN(model, X) -> integer class ids 0..6        (both injected by the torch cell)
FIT_FN, PREDICT_FN = None, None

def run_one(cell, seed, rid, out_dir, epochs_override=None, save=True):
    t0 = time.time(); target, B, states = cell['target'], cell['budget'], tuple(cell['states'])
    split = draw_split(target, B, states, seed, POOLS, USA_CROP)
    chk = verify_split(split, target, B, states, USA_STATE, USA_CROP)
    if not chk['ok']: raise IntegrityFailure('SPLIT NOT EXACT: ' + '; '.join(chk['msgs']))
    tr, va = split['train_pos'], split['val_pos']
    out = FIT_FN(take(FEATURES['USA'], tr), USA_CROP_ID[tr], take(FEATURES['USA'], va), USA_CROP_ID[va], seed, epochs_override)
    preds = np.asarray(PREDICT_FN(out['model'], EVAL_X[target])).astype(int); y_true = np.asarray(EVAL_Y[target]).astype(int)
    train_counts = np.bincount(USA_CROP_ID[tr], minlength=7); val_counts = np.bincount(USA_CROP_ID[va], minlength=7)
    sc = score_run(target, y_true, preds, train_counts)
    I, ok = evaluate_integrity(sc, preds, chk, out['info']); skills = skill_flags(sc, out['info'])
    row = build_row(rid, cell, seed, split, sc, I, ok, skills, out['info'], preds, y_true, time.time() - t0, USA_CROP)
    if save:                                                                    # 1) per-run arrays (atomic)   2) the metrics row is appended by the caller LAST
        def w(f):
            np.savez_compressed(f, run_id=rid, y_true=y_true.astype(np.uint8), y_pred=preds.astype(np.uint8), y_pred_paper_rule=sc['yp_remapped'].astype(np.uint8),
                                target_sample_ids=np.asarray(TARGET_IDS[target]).astype(np.int64), usa_train_ids=USA_NPZ_ROW[tr].astype(np.int64), usa_val_ids=USA_NPZ_ROW[va].astype(np.int64),
                                train_counts=train_counts, val_counts=val_counts, cm7=sc['cm7'], cm_paper_rule=sc['cm_paper'],
                                cm_binary=(sc['cm_binary'] if sc['cm_binary'] is not None else np.zeros((0, 0), dtype=int)),
                                paper_precision=sc['paper_prec'], paper_recall=sc['paper_rec'], paper_f1_per_class=sc['paper_f1_cls'],
                                binary_precision=(sc['bin_prec'] if sc['bin_prec'] is not None else np.zeros(0)), binary_recall=(sc['bin_rec'] if sc['bin_rec'] is not None else np.zeros(0)),
                                binary_f1_per_class=(sc['bin_f1_cls'] if sc['bin_f1_cls'] is not None else np.zeros(0)), val_history=np.asarray(out['info'].get('history', []), dtype=np.float32))
        atomic_write(os.path.join(out_dir, 'runs', hashlib.md5(rid.encode()).hexdigest()[:16] + '.npz'), w)
    return row, preds, y_true, sc

def take(X, idx):
    idx = np.asarray(idx)
    if type(X).__module__.startswith('torch'):
        import torch
        return X[torch.as_tensor(idx, dtype=torch.long)]
    return X[idx]

def _log(msg, path=None):
    print(msg, flush=True)
    if path:
        with open(path, 'a') as f: f.write(msg + '\n')

def _pct(v): return '  n/a' if v is None or not np.isfinite(v) else f'{100 * v:5.1f}'

def seed_line(row, sc):
    sk = row['skill_flags']
    return (f"  [{row['target']} B{row['budget']} k{row['k']} {row['states']} s{row['seed']}] {row['primary_view']} mF1 {_pct(row['primary_f1'])} "
            f"(all-other {_pct(row['baseline_all_other_primary'])}, majority {_pct(row['baseline_majority_primary'])}) | paper-rule mF1 {_pct(row['paper_f1'])} OA {_pct(row['paper_oa'])} "
            f"| cov-cond {_pct(row['cov_cond_primary_f1'])} | val {_pct(row['val_f1'])} best-ep {row['best_epoch']} ran {row['epochs_run']}/{MAX_EPOCHS} ({row['stop_reason']}) "
            f"| INTEGRITY {'PASS' if row['integrity_pass'] else 'FAIL'}" + (f' | {sk}' if sk else ''))

def run_seed(seed, cells, results_csv, out_dir, log_path=None, epochs_override=None):
    # one seed over ALL planned cells; resumable (finished runs are skipped); aborts only after MAX_CONSEC_INTEGRITY consecutive integrity failures/errors
    prev = load_results(results_csv); done = set(prev.loc[prev['status'].isin(['ok', 'integrity_fail']), 'run_id']) if len(prev) else set()
    todo = [c for c in cells if make_run_id(c, seed) not in done]
    _log(f'SEED {seed}: {len(cells)} planned runs | already done {len(cells) - len(todo)} | pending {len(todo)}', log_path)
    t_start, n_new, consec = time.time(), 0, 0
    for cell in todo:
        rid = make_run_id(cell, seed)
        try:
            row, preds, y_true, sc = run_one(cell, seed, rid, out_dir, epochs_override=epochs_override)
        except TooFewClasses: raise                                              # target data problem: stop the whole run
        except Exception as e:
            consec += 1; msg = f'{type(e).__name__}: {e}'[:400]; _log(f'  !! INTEGRITY/ERROR in {rid}: {msg}', log_path)
            with open(os.path.join(out_dir, 'errors.log'), 'a') as f: f.write(f'{rid}\n{traceback.format_exc()}\n')
            er = {c: '' for c in COLUMNS}; er.update(run_id=rid, status='integrity_fail' if isinstance(e, IntegrityFailure) else 'error', target=cell['target'], budget=cell['budget'], k=cell['k'],
                                                     combo_id=cell['combo_id'], states='-'.join(cell['states']), seed=seed, error=msg, integrity_pass=False)
            append_row(results_csv, er)
            if consec >= MAX_CONSEC_INTEGRITY: raise RuntimeError(f'{consec} consecutive integrity failures/errors -- aborted. Last: {msg}')
            continue
        append_row(results_csv, row); n_new += 1                                 # the metrics file is written LAST
        _log(seed_line(row, sc), log_path)
        consec = consec + 1 if row['status'] == 'integrity_fail' else 0
        if consec >= MAX_CONSEC_INTEGRITY: raise RuntimeError(f'{consec} consecutive integrity failures -- aborted (see the lines above).')
        if n_new % 10 == 0:
            el = time.time() - t_start; _log(f'  .. {n_new} new runs, {el / n_new:.1f}s/run, ~{(len(todo) - n_new) * el / n_new / 3600:.1f} h left for this seed', log_path)
    _log(f'SEED {seed} COMPLETE: {n_new} new runs this session', log_path)
    return 'complete'

# ------------------------------------------------------------------------------ STATISTICS (spec 5)
def t_interval_fpc(x, N, level=0.95):
    # t-interval over state COMBINATIONS with finite-population correction sqrt((N-n)/(N-1)); N = all C(pool,k) combinations
    x = np.asarray(x, dtype=float); x = x[np.isfinite(x)]; n = len(x); m = float(x.mean()) if n else float('nan')
    if n < 2: return m, float('nan'), float('nan'), float('nan')
    fpc = math.sqrt(max(N - n, 0) / (N - 1)) if N > 1 else 0.0
    se = float(x.std(ddof=1)) / math.sqrt(n) * fpc; h = float(scipy_stats.t.ppf(0.5 + level / 2, n - 1)) * se
    return m, m - h, m + h, se

def combo_level(df, metric='primary_f1'):
    # unit = state combination; the seeds are averaged FIRST
    d = df[df['status'] == 'ok'].copy()
    for c in [metric, 'budget', 'k', 'seed', 'pool_size', 'n_total_combos', 'key_n']: d[c] = pd.to_numeric(d[c], errors='coerce')
    g = d.groupby(['target', 'budget', 'k', 'states']).agg(f1=(metric, 'mean'), seed_sd=(metric, lambda x: float(np.std(x, ddof=1)) if len(x) > 1 else float('nan')),
                                                         n_seeds=(metric, 'count'), pool_size=('pool_size', 'first'), n_total=('n_total_combos', 'first'), key_n=('key_n', 'first')).reset_index()
    return g

def jackknife_states(cl_group):
    # delete-one-STATE jackknife over the states that appear in the evaluated combinations (secondary generalisation interval)
    combos = [(set(s.split('-')), f) for s, f in zip(cl_group['states'], cl_group['f1'])]; states = sorted(set().union(*[c for c, _ in combos])); thetas = []
    for s in states:
        rest = [f for c, f in combos if s not in c]
        if rest: thetas.append(float(np.mean(rest)))
    m = len(thetas)
    if m < 2: return float('nan'), float('nan'), float('nan')
    tb = float(np.mean(thetas)); se = math.sqrt((m - 1) / m * sum((t - tb) ** 2 for t in thetas)); h = float(scipy_stats.t.ppf(0.975, m - 1)) * se
    return se, float(np.mean([f for _, f in combos])) - h, float(np.mean([f for _, f in combos])) + h

def cell_stats(df, metric='primary_f1'):
    cl = combo_level(df, metric); rows = []
    for (t, B, k), g in cl.groupby(['target', 'budget', 'k']):
        x = g['f1'].values; N = int(g['n_total'].iloc[0]) if pd.notna(g['n_total'].iloc[0]) else len(x); census = (k == 1)
        m, lo, hi, se = t_interval_fpc(x, N) if not census else (float(np.mean(x)), float('nan'), float('nan'), float('nan'))
        jse, jlo, jhi = jackknife_states(g)
        sub = df[(df['status'] == 'ok') & (df['target'] == t) & (df['budget'] == B) & (df['k'] == k)]
        per_seed_means = sub.groupby('seed')[metric].mean().values if len(sub) else np.array([])
        rows.append(dict(target=t, budget=int(B), k=int(k), n_combos=len(x), N_total_combos=N, n_seeds_min=int(g['n_seeds'].min()), mean=m, ci_lo=lo, ci_hi=hi, se=se, census=census,
                         census_values=';'.join(f'{s}:{f:.4f}' for s, f in zip(g['states'], g['f1'])) if census else '',
                         jackknife_se=jse, jackknife_lo=jlo, jackknife_hi=jhi,
                         seed_sd_within_combo=float(np.nanmean(g['seed_sd'])) if g['seed_sd'].notna().any() else float('nan'),
                         seed_sd_of_cell_means=float(np.std(per_seed_means, ddof=1)) if len(per_seed_means) > 1 else float('nan')))
    return pd.DataFrame(rows).sort_values(['target', 'budget', 'k']).reset_index(drop=True) if rows else pd.DataFrame()

def gain_kmin_kmax(cs):
    # gain from the smallest to the largest feasible k with a CI (Welch-style; a k=1 census contributes zero sampling variance)
    out = []
    for (t, B), g in cs.groupby(['target', 'budget']):
        g = g.sort_values('k')
        if len(g) < 2: continue
        lo, hi = g.iloc[0], g.iloc[-1]; d = float(hi['mean'] - lo['mean'])
        se2 = [s ** 2 for s in (lo['se'], hi['se']) if np.isfinite(s)]; ns = [n for s, n in ((lo['se'], lo['n_combos']), (hi['se'], hi['n_combos'])) if np.isfinite(s)]
        if se2 and sum(se2) > 0:
            se = math.sqrt(sum(se2)); df_w = (sum(se2) ** 2) / sum((s2 ** 2) / (n - 1) for s2, n in zip(se2, ns)); h = float(scipy_stats.t.ppf(0.975, df_w)) * se
        else: se, h = float('nan'), float('nan')
        out.append(dict(target=t, budget=int(B), k_min=int(lo['k']), k_max=int(hi['k']), gain=d, ci_lo=d - h if np.isfinite(h) else float('nan'), ci_hi=d + h if np.isfinite(h) else float('nan')))
    return pd.DataFrame(out)

def regression_report(df, metric='primary_f1'):
    # OLS of combination-level F1 on k, key-crop count and budget, per target.  NB: key-crop count = share x budget EXACTLY -> aliased with budget (reported, not hidden).
    cl = combo_level(df, metric); res = []
    for t, g in cl.groupby('target'):
        X_all = np.column_stack([np.ones(len(g)), g['k'].values.astype(float), g['key_n'].values.astype(float), g['budget'].values.astype(float)]); names = ['intercept', 'k', 'key_crop_count', 'budget']
        keep = list(range(4)); note = ''
        if np.linalg.matrix_rank(X_all) < 4:
            keep = [0, 1, 3]; note = 'key_crop_count is perfectly collinear with budget (= share x budget) -> dropped; its effect is NOT separable from budget'
        X = X_all[:, keep]; y = g['f1'].values.astype(float)
        if len(y) <= X.shape[1] or np.linalg.matrix_rank(X) < X.shape[1]:
            res.append(dict(target=t, term='(not estimable)', coef=np.nan, se=np.nan, t=np.nan, p=np.nan, n=len(y), r2=np.nan, note=note + ' | design matrix rank-deficient or too few rows')); continue
        beta, *_ = np.linalg.lstsq(X, y, rcond=None); resid = y - X @ beta; dfree = len(y) - X.shape[1]; s2 = float(resid @ resid) / dfree
        cov = s2 * np.linalg.inv(X.T @ X); se = np.sqrt(np.diag(cov)); tv = beta / se; pv = 2 * scipy_stats.t.sf(np.abs(tv), dfree)
        r2 = 1 - float(resid @ resid) / float(((y - y.mean()) ** 2).sum()) if ((y - y.mean()) ** 2).sum() > 0 else float('nan')
        for i, j in enumerate(keep): res.append(dict(target=t, term=names[j], coef=float(beta[i]), se=float(se[i]), t=float(tv[i]), p=float(pv[i]), n=len(y), r2=r2, note=note))
    return pd.DataFrame(res)

def seed_direction_warnings(df, metric='primary_f1'):
    # WARNING only, never a stop: do the seeds disagree on the SIGN of the k_min -> k_max change?
    w = []; d = df[df['status'] == 'ok'].copy()
    for c in [metric, 'budget', 'k', 'seed']: d[c] = pd.to_numeric(d[c], errors='coerce')
    for (t, B), g in d.groupby(['target', 'budget']):
        ks = sorted(g['k'].unique())
        if len(ks) < 2: continue
        signs = {}
        for s, gs in g.groupby('seed'):
            a, b = gs[gs['k'] == ks[0]][metric].mean(), gs[gs['k'] == ks[-1]][metric].mean()
            if np.isfinite(a) and np.isfinite(b): signs[int(s)] = float(np.sign(b - a))
        if len(set(signs.values())) > 1: w.append(f'WARNING seed-direction disagreement [{t} B{int(B)} k{int(ks[0])}->k{int(ks[-1])}]: per-seed sign {signs}')
    return w

def gate_report(df, seed, cells):
    # the per-seed gate report you approve before the next seed
    d = df[pd.to_numeric(df['seed'], errors='coerce') == seed]; L = []; planned = len(cells)
    L.append(f'================ GATE REPORT  seed {seed}  ({EXPERIMENT_VERSION}) ================')
    L.append(f'planned runs {planned} | recorded {len(d)} | status {d["status"].value_counts().to_dict()}')
    L.append('-- INTEGRITY (abort-level) --')
    for c in INTEGRITY_CHECKS:
        bad = int((d[c].astype(str) != 'True').sum()) if len(d) else 0; L.append(f'   {c:28s} failures: {bad}')
    rd = pd.to_numeric(d['restore_delta'], errors='coerce'); sd = pd.to_numeric(d['scorer_maxdiff'], errors='coerce')
    L.append(f'   max restore delta {rd.max():.2e} (tolerance {RESTORE_TOL:.0e}) | max scorer disagreement {sd.max():.2e} (tolerance {SCORER_TOL:.0e})')
    L.append('-- TRAINING --'); L.append(f'   stop reasons: {d["stop_reason"].value_counts().to_dict()}')
    be = pd.to_numeric(d['best_epoch'], errors='coerce'); er = pd.to_numeric(d['epochs_run'], errors='coerce')
    L.append(f'   epochs run: min {er.min():.0f} median {er.median():.0f} max {er.max():.0f} (cap {MAX_EPOCHS}) | best_epoch >= {UNDERTRAINED_EPOCH} (possibly_undertrained): {int((be >= UNDERTRAINED_EPOCH).sum())}')
    L.append('-- SKILL outcomes (warnings, never aborts) --')
    flags = d['skill_flags'].fillna('').astype(str).str.split('|').explode().replace('', np.nan).dropna()
    L.append(f'   flags: {flags.str.replace(r":.*", "", regex=True).value_counts().to_dict()}')
    for t in [x for x in TARGETS_ORDER if x in set(d['target'])]:
        g = d[d['target'] == t].copy()
        for c in ['primary_f1', 'baseline_all_other_primary', 'baseline_majority_primary', 'paper_f1', 'val_f1']: g[c] = pd.to_numeric(g[c], errors='coerce')
        L.append(f'   {t} ({PRIMARY_VIEW[t][0]}{"_" + PRIMARY_VIEW[t][1] if PRIMARY_VIEW[t][1] else ""}): primary mean {100 * g.primary_f1.mean():.1f} | all-other {100 * g.baseline_all_other_primary.mean():.1f} | '
                 f'majority {100 * g.baseline_majority_primary.mean():.1f} | paper-rule mean {100 * g.paper_f1.mean():.1f} | val mF1 min {100 * g.val_f1.min():.1f} | '
                 f'runs not better than baseline: {int((g.primary_f1 <= g[["baseline_all_other_primary", "baseline_majority_primary"]].max(axis=1)).sum())}/{len(g)}')
    cs = cell_stats(df[pd.to_numeric(df['seed'], errors='coerce') == seed])
    if len(cs):
        L.append('-- primary mean by (target, budget, k) for THIS seed (never compare binary F1 with multiclass F1 across targets) --')
        for t, gt in cs.groupby('target'):
            L.append(f'   {t}: ' + ' | '.join(f'B{int(r.budget)}k{int(r.k)}:{100 * r["mean"]:.1f}' for _, r in gt.iterrows()))
    L += seed_direction_warnings(df)
    return '\n'.join(L)

## Test gate - Part A (no GPU, no repo, no data)

In [ ]:
# ======================================================================================
# PRE-TRAINING TEST GATE, PART A (spec section 8).  No GPU, no repo, no real data.  Training cells refuse to run unless Part A AND Part B pass.
# ======================================================================================
from sklearn.linear_model import LogisticRegression
TEST_REPORT = []
def _t(name):
    def deco(fn):
        try: fn(); TEST_REPORT.append((name, 'PASS', ''))
        except Exception as e: TEST_REPORT.append((name, 'FAIL', f'{type(e).__name__}: {e}'))
        return fn
    return deco
def _near(a, b, tol=1e-9): assert abs(a - b) <= tol, f'{a} != {b}'
def _ids(*pairs): return np.concatenate([np.full(n, CROP_ID[c]) for c, n in pairs])
def _labels_from_table2(t): return np.concatenate([np.full(n, CROP_ID[c]) for c, n in TABLE2[t].items() if n > 0])

@_t('Table 2 label sets: ARG/CHN/AUS/FRA exactly as the paper')
def _():
    assert DEFINED_CLASSES['ARG'] == ['corn', 'soybeans', 'rice', 'sugarcane', 'cotton', 'other']
    assert DEFINED_CLASSES['CHN'] == ['corn', 'soybeans', 'rice', 'other'] and DEFINED_CLASSES['AUS'] == ['sugarcane', 'cotton', 'other']
    assert DEFINED_CLASSES['FRA'] == ['corn', 'soybeans', 'rice', 'wheat', 'other']

@_t('binary_collapse on the REAL label sets: other = every non-positive class (true OR predicted); nothing dropped, nothing -1')
def _():
    for t, pos in BINARY_TARGETS.items():
        yt = _labels_from_table2(t); rng = np.random.default_rng(0); yp = rng.integers(0, 7, len(yt))
        bt, bp = binary_collapse(yt, yp, pos)
        assert len(bt) == len(yt) and len(bp) == len(yp) and set(np.unique(bt)) <= {0, 1} and set(np.unique(bp)) <= {0, 1} and bt.min() >= 0 and bp.min() >= 0
        assert (bt == (yt == CROP_ID[pos])).all() and (bp == (yp == CROP_ID[pos])).all()
    # spec example "true corn predicted soybeans":  CHN (positive=rice) -> other/other (a correct 'other');  ARG (positive=soybeans) -> other/POSITIVE (a false positive)
    a, b = binary_collapse([CROP_ID['corn']], [CROP_ID['soybeans']], 'rice'); assert (a[0], b[0]) == (0, 0)
    a, b = binary_collapse([CROP_ID['corn']], [CROP_ID['soybeans']], 'soybeans'); assert (a[0], b[0]) == (0, 1)
    a, b = binary_collapse([CROP_ID['wheat'], CROP_ID['cotton']], [CROP_ID['wheat'], CROP_ID['rice']], 'rice'); assert list(a) == [0, 0] and list(b) == [0, 1]
    # a true crop outside {positive, other} is allowed (never required to be a subset)
    sc = score_run('CHN', _ids(('corn', 5), ('rice', 5), ('other', 5)), _ids(('corn', 5), ('rice', 5), ('other', 5)), np.array([50, 50, 50, 0, 0, 0, 50])); _near(sc['primary_f1'], 1.0)

@_t("all-'other' baseline reproduces 0.4375 (CHN, rice view) and 0.4496 (ARG, soybeans view) from the Table 2 counts")
def _():
    for t, exp in (('CHN', 0.4375), ('ARG', 0.4496)):
        yt = _labels_from_table2(t); sc = score_run(t, yt, np.full(len(yt), CROP_ID['other']), np.array([100] * 7))
        assert round(sc['baseline_all_other_primary'], 4) == exp, (t, sc['baseline_all_other_primary'])
        assert round(sc['primary_f1'], 4) == exp and sc['primary_view'] == f'binary_{BINARY_TARGETS[t]}'
        assert len(yt) == TABLE2_TOTAL[t]

def _authors_remap_reference(y_pred, target):
    # INDEPENDENT re-implementation of the spec text (lookup tables, no np.isin) -- compared with remap_paper_rule
    out = []
    for y in np.asarray(y_pred).tolist():
        n = CROP_LABELS[y]
        if target == 'AUS': out.append(CROP_ID[n] if n in ('sugarcane', 'cotton') else CROP_ID['other'])
        elif target == 'CHN': out.append(CROP_ID[n] if n in ('corn', 'soybeans', 'rice') else CROP_ID['other'])
        elif target == 'ARG': out.append(CROP_ID['other'] if n == 'wheat' else CROP_ID[n])
        else: out.append(CROP_ID[n])
    return np.array(out)

@_t('paper-rule scorer == macro f1_score(y_true, remapped, labels=np.unique(y_true)) on random data, all targets (reference remap)')
def _():
    rng = np.random.default_rng(1)
    for t in ['ARG', 'CHN', 'AUS', 'FRA']:
        for _rep in range(5):
            yt = rng.choice([CROP_ID[c] for c in DEFINED_CLASSES[t]], 3000, p=None); yp = rng.integers(0, 7, 3000)
            assert (remap_paper_rule(yp, t) == _authors_remap_reference(yp, t)).all()
            sc = score_run(t, yt, yp, np.array([100] * 7)); yr = _authors_remap_reference(yp, t)
            _near(sc['paper_f1'], f1_score(yt, yr, average='macro', labels=np.unique(yt)), 1e-12); _near(sc['paper_oa'], accuracy_score(yt, yr), 1e-12)

@_t('FRA: NO remap (wrong crop stays an error); ARG: only wheat->other; CHN keeps corn/soy/rice; AUS keeps sugarcane/cotton')
def _():
    allc = np.arange(7)
    assert (remap_paper_rule(allc, 'FRA') == allc).all()
    assert list(remap_paper_rule(allc, 'ARG')) == [0, 1, 2, 6, 4, 5, 6]
    assert list(remap_paper_rule(allc, 'CHN')) == [0, 1, 2, 6, 6, 6, 6] and list(remap_paper_rule(allc, 'AUS')) == [6, 6, 6, 6, 4, 5, 6]

@_t('two independent scorers (sklearn vs from-scratch numpy) agree to 1e-12 on random data (paper rule + binary view, per-class P/R/F1)')
def _():
    rng = np.random.default_rng(2)
    for t in ['ARG', 'CHN', 'AUS', 'FRA']:
        for _rep in range(8):
            n = int(rng.integers(50, 4000)); yt = rng.choice([CROP_ID[c] for c in DEFINED_CLASSES[t]], n); yp = rng.integers(0, 7, n)
            tc = rng.integers(0, 400, 7); sc = score_run(t, yt, yp, tc); assert sc['scorer_maxdiff'] <= SCORER_TOL, sc['scorer_maxdiff']

@_t('stop rule: fewer than 2 true classes raises TooFewClasses')
def _():
    try: score_run('ARG', _ids(('soybeans', 20)), _ids(('soybeans', 20)), np.ones(7)); raise RuntimeError('no stop')
    except TooFewClasses: pass

@_t('classes absent from training stay IN the score at their F1 (flagged); coverage-conditional macro-F1 is separate and labelled')
def _():
    yt = _ids(('corn', 50), ('soybeans', 50), ('rice', 50), ('sugarcane', 50), ('cotton', 50), ('other', 50)); yp = yt.copy()
    tc = np.array([100, 100, 100, 0, 0, 0, 100])                    # cotton / sugarcane absent from training
    yp[yt == CROP_ID['sugarcane']] = CROP_ID['other']
    sc = score_run('ARG', yt, yp, tc)
    assert 'sugarcane' in sc['absent_train_classes'] and 'cotton' in sc['absent_train_classes']
    assert sc['paper_f1_cls'][CROP_ID['sugarcane']] == 0.0 and not np.isnan(sc['paper_f1_cls'][CROP_ID['cotton']])      # still scored
    assert sc['cov_cond_paper_classes'] == ['corn', 'soybeans', 'rice', 'other'] and sc['cov_cond_paper_f1'] != sc['paper_f1']

@_t('no 7-class macro over crops the target lacks: per-class F1 is NaN for classes absent from y_true; macro only over np.unique(y_true)')
def _():
    yt = _ids(('rice', 30), ('other', 70)); sc = score_run('CHN', yt, yt.copy(), np.array([100] * 7))
    assert sc['present'] == ['rice', 'other'] and np.isnan(sc['paper_f1_cls'][CROP_ID['corn']]) and sc['paper_f1'] == 1.0

@_t('integrity vs skill: a skill failure (low val F1 / no skill / collapsed) is a WARNING; only integrity checks can fail the run')
def _():
    yt = _ids(('soybeans', 50), ('other', 50)); allo = np.full(100, CROP_ID['other']); sc = score_run('ARG', yt, allo, np.array([100] * 7))
    info = dict(val_f1=0.3, restore_delta=0.0, best_epoch=47); I, ok = evaluate_integrity(sc, allo, dict(ok=True), info); fl = skill_flags(sc, info)
    assert ok and all(I.values()); assert any(f.startswith('WARN_VAL') for f in fl) and 'WARN_NOT_BETTER_THAN_BASELINE' in fl and 'WARN_COLLAPSED_PREDICTIONS' in fl and 'possibly_undertrained' in fl
    I2, ok2 = evaluate_integrity(sc, allo, dict(ok=False), info); assert not ok2 and not I2['I3_split_exact']
    I3, ok3 = evaluate_integrity(sc, allo, dict(ok=True), dict(val_f1=0.9, restore_delta=1e-3)); assert not ok3 and not I3['I4_restore_matches_best']
    I4, ok4 = evaluate_integrity(sc, allo, dict(ok=True), dict(val_f1=float('nan'), restore_delta=0.0)); assert not ok4 and not I4['I5_finite']
    bad = np.full(100, 9); 
    try: score_run('ARG', yt, bad, np.ones(7)); raise RuntimeError('no stop')
    except AssertionError: pass

@_t('Table 3 reference: only the three published values, no too-good flag, no PAPER_REF_F1')
def _():
    assert TABLE3_REF_MF1 == {'ARG': 75.63, 'AUS': 88.38, 'FRA': 90.00} and 'PAPER_REF_F1' not in globals()
    assert not any('TOO_GOOD' in f for f in skill_flags(score_run('ARG', _ids(('soybeans', 50), ('other', 50)), _ids(('soybeans', 50), ('other', 50)), np.ones(7) * 100), dict(val_f1=1.0, best_epoch=3)))

@_t('early stopping: strictly-greater improvement, patience 15, min 20 epochs, hard cap 50, reasons logged')
def _():
    es = EarlyStopper(); stops = None
    for i in range(100):                                            # never improves after epoch 1
        _imp, stop = es.step(0.5)
        if stop: stops = es.epoch; break
    assert stops == 20 and es.reason == 'early_stop_patience' and es.best_epoch == 1          # min epochs 20 respected, bad count already >= 15
    es = EarlyStopper(); seq = [0.1 * i / 10 for i in range(1, 51)]                          # improves every epoch -> runs to the cap
    for s in seq:
        _imp, stop = es.step(s)
        if stop: break
    assert es.epoch == 50 and es.reason == 'max_epochs' and es.best_epoch == 50
    es = EarlyStopper()
    for s in [0.5] + [0.5] * 30:                                    # equal score is NOT an improvement (strictly greater)
        _imp, stop = es.step(s)
        if stop: break
    assert es.best_epoch == 1 and es.epoch == 20
    es = EarlyStopper()                                              # best at epoch 10 -> stops at 25
    for i in range(60):
        _imp, stop = es.step(0.9 if i == 9 else 0.5)
        if stop: break
    assert es.epoch == 25 and es.best_epoch == 10
    try: EarlyStopper(max_epochs=51); raise RuntimeError('no stop')
    except AssertionError: pass

@_t('class weights: absent crops get weight 0 (no division by zero) and match sklearn on the present classes')
def _():
    from sklearn.utils.class_weight import compute_class_weight
    y = np.array([1] * 30 + [6] * 10 + [2] * 60); w = balanced_class_weights(y, 7)
    assert w[0] == w[3] == w[4] == w[5] == 0 and np.isfinite(w).all() and np.allclose(w[[1, 2, 6]], compute_class_weight('balanced', classes=np.array([1, 2, 6]), y=y))

def _synthetic_usa(n_states=14, rows=1100, seed=0):
    rng = np.random.default_rng(seed); st, cr = [], []
    for i in range(n_states):
        p = rng.dirichlet(np.ones(7) * 1.5); p = 0.04 + p; p /= p.sum()
        for _ in range(rows): st.append('ABCDEFGHIJKLMNOPQRSTUVWXYZ'[i]); cr.append(CROP_LABELS[int(rng.choice(7, p=p))])
    return np.array(st), np.array(cr)

@_t('design: key counts exact (ARG soy 20% / CHN rice 20% / FRA 15% x3), identical at every k; pool = states that can supply the whole budget alone')
def _():
    for B in BUDGETS:
        assert key_counts('ARG', B) == {'soybeans': int(0.2 * B)} and key_counts('CHN', B) == {'rice': int(0.2 * B)}
        assert key_counts('FRA', B) == {'corn': int(0.15 * B), 'soybeans': int(0.15 * B), 'wheat': int(0.15 * B)} and rest_count('FRA', B) == B - 3 * int(0.15 * B)
    cap = pd.DataFrame({c: [900, 900, 900] for c in CROP_ORDER}, index=['X', 'Y', 'Z']); cap['total'] = cap[CROP_ORDER].sum(axis=1)
    cap.loc['X', 'soybeans'] = 100; cap.loc['Y', 'total'] = 1400                                    # X: too few soybeans for B=1000 (needs 200) ; Y: below the 1500-row rule
    assert eligible_states('ARG', 1000, cap) == ['Z'] and 'X' not in eligible_states('ARG', 1000, cap)

@_t('plan rules: k=1 census of the pool; k>=2 <=10 deterministic combos; <5 combos or <3 k-points -> infeasible; CHN k<=4; FRA k=8 needs pool>=10; never shrinks a budget')
def _():
    st, cr = _synthetic_usa(14, 1800); cap = make_cap(st, cr); cells, grid = build_plan(cap)
    g = grid[(grid.target == 'ARG') & (grid.budget == 1000)].set_index('k'); P = int(g.pool_size.iloc[0]); pool = eligible_states('ARG', 1000, cap); assert P == len(pool)
    assert g.loc[1, 'n_combos'] == P and all(g.loc[k, 'n_combos'] <= MAX_COMBOS for k in g.index if k >= 2)
    assert not grid[(grid.target == 'CHN') & (grid.k == 8)].feasible.any()
    c1 = make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 1000); assert len(c1) == len(set(c1)) == MAX_COMBOS and c1 == make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 1000)
    assert make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 1000) != make_combos(list('ABCDEFGHIJKL'), 4, 'ARG', 2000)
    small = st != 'Z'; s2, c2 = st[:], cr[:]                                                         # force a tiny pool -> everything infeasible
    cap_small = make_cap(*_synthetic_usa(4, 1800)); _c, gr = build_plan(cap_small)
    assert not gr[(gr.target == 'FRA') & (gr.k == 8)].feasible.any()
    only = gr[(gr.target == 'ARG') & (gr.budget == 1000)]; assert (only.feasible.sum() == 0) or (only.feasible.sum() >= MIN_KPOINTS)
    assert all(c['pool_size'] >= 1 for c in cells) and set(c['budget'] for c in cells) <= set(BUDGETS)
    aus = [c for c in cells if c['target'] == 'AUS']; assert all(c['pending'] for c in aus) == (not AUS_SPEC_CONFIRMED)

@_t('sampling: exact budget, SAME key-crop count at k=1,2,4,8, 10% stratified validation, unique ids, no overlap, deterministic, seed-dependent')
def _():
    st, cr = _synthetic_usa(14, 1800); cap = make_cap(st, cr); pools = build_pools(st, cr)
    for t in ['ARG', 'CHN', 'FRA']:
        for B in [1000, 4000]:
            pool = eligible_states(t, B, cap)
            if len(pool) < 8: continue
            ref = None
            for k in [1, 2, 4, 8]:
                states = tuple(pool[:k]); sp = draw_split(t, B, states, 0, pools, cr); v = verify_split(sp, t, B, states, st, cr); assert v['ok'], v['msgs']
                allp = np.concatenate([sp['train_pos'], sp['val_pos']]); kc = {c: int((cr[allp] == c).sum()) for c in key_counts(t, B)}
                assert len(sp['val_pos']) == B // 10 and len(allp) == B and len(np.unique(allp)) == B
                ref = ref or kc; assert kc == ref == key_counts(t, B), 'key-crop count changed with k'
    t, B = 'ARG', 1000; pool = eligible_states(t, B, cap); a = draw_split(t, B, tuple(pool[:2]), 0, pools, cr); b = draw_split(t, B, tuple(pool[:2]), 0, pools, cr); c = draw_split(t, B, tuple(pool[:2]), 1, pools, cr)
    assert (a['train_pos'] == b['train_pos']).all() and not np.array_equal(np.sort(a['train_pos']), np.sort(c['train_pos']))
    bad = dict(a); bad['val_pos'] = a['train_pos'][:100]; assert not verify_split(bad, t, B, tuple(pool[:2]), st, cr)['ok']            # overlap is caught
    bad2 = dict(a); bad2['train_pos'] = np.concatenate([a['train_pos'][:-1], a['train_pos'][:1]]); assert not verify_split(bad2, t, B, tuple(pool[:2]), st, cr)['ok']   # duplicate id is caught

@_t('statistics: FPC interval collapses when all combinations are evaluated; k=1 census has no CI; jackknife defined; regression reports the key-count/budget aliasing')
def _():
    x = np.array([0.5, 0.6, 0.7, 0.55, 0.65]); m, lo, hi, se = t_interval_fpc(x, 5); assert se == 0 and lo == hi == m
    m2, lo2, hi2, se2 = t_interval_fpc(x, 100); assert lo2 < m2 < hi2 and se2 > 0
    rows = []
    for t, kc in (('ARG', 200), ('ARG', 400)):
        for k in (1, 2, 4):
            for ci in range(6):
                for s in range(2): rows.append(dict(run_id=f'{t}{kc}{k}{ci}{s}', status='ok', target=t, budget=kc * 5, k=k, states='-'.join(f'S{(ci + j) % 8}' for j in range(k)), seed=s, primary_f1=0.5 + 0.01 * k + 0.0001 * ci + 0.01 * s,
                                                  pool_size=8, n_total_combos=math.comb(8, k), key_n=kc))
    df = pd.DataFrame(rows); cs = cell_stats(df); assert (cs[cs.k == 1].ci_lo.isna()).all() and cs[cs.k == 1].census.all() and cs.jackknife_se.notna().all()
    reg = regression_report(df); assert 'perfectly collinear' in ' '.join(reg['note'].astype(str)) and 'key_crop_count' not in set(reg.term)
    gn = gain_kmin_kmax(cs); assert len(gn) == 2 and (gn.gain > 0).all()

@_t('atomic write: no temp files left behind; content replaced as a whole')
def _():
    with tempfile.TemporaryDirectory() as td:
        p = os.path.join(td, 'x.csv'); atomic_write(p, lambda f: f.write('a,b\n1,2\n'), mode='w'); atomic_write(p, lambda f: f.write('a,b\n3,4\n'), mode='w')
        assert open(p).read() == 'a,b\n3,4\n' and os.listdir(td) == ['x.csv']

def selftest_pipeline():
    # END-TO-END dry run with a stand-in model (logistic regression): plan -> draw -> verify -> fit -> predict -> score (2 scorers) -> gates -> npz -> CSV (last) -> resume -> stats -> gate report.
    names = ['BUDGETS', 'SEEDS', 'K_LIST', 'MAX_COMBOS', 'MIN_STATE_ROWS', 'AUS_SPEC_CONFIRMED', 'AUS_STATES', 'AUS_KEY_SHARES_IF_CONFIRMED', 'FRA_K8_MIN_POOL']
    g = globals(); saved_cfg = {n: g[n] for n in names}
    keys = ['FEATURES', 'USA_STATE', 'USA_CROP', 'USA_CROP_ID', 'USA_NPZ_ROW', 'POOLS', 'EVAL_X', 'EVAL_Y', 'TARGET_IDS', 'FIT_FN', 'PREDICT_FN']; saved = {n: g.get(n, '<unset>') for n in keys}
    try:
        g.update(BUDGETS=[200, 400], SEEDS=[0, 1], K_LIST=[1, 2, 4, 8], MIN_STATE_ROWS=300, AUS_SPEC_CONFIRMED=True, AUS_STATES=['A', 'B'], AUS_KEY_SHARES_IF_CONFIRMED={'sugarcane': 0.2}, FRA_K8_MIN_POOL=10)
        st, cr = _synthetic_usa(14, 700); yid = np.array([CROP_ID[c] for c in cr]); rng = np.random.default_rng(0); mu = rng.normal(0, 1, (7, 730)).astype(np.float32)
        def feats(y): return (mu[y] + rng.normal(0, 1.5, (len(y), 730))).astype(np.float32).reshape(len(y), 1, 10, 73)
        g.update(FEATURES={'USA': feats(yid)}, USA_STATE=st, USA_CROP=cr, USA_CROP_ID=yid, USA_NPZ_ROW=np.arange(len(st)) * 2, POOLS=build_pools(st, cr))
        EX, EY, TI = {}, {}, {}
        for t in TABLE2:
            if t == 'USA': continue
            y = np.array([CROP_ID[c] for c in DEFINED_CLASSES[t] for _ in range(60)]); EY[t] = y; EX[t] = feats(y); TI[t] = np.arange(len(y))
        g.update(EVAL_X=EX, EVAL_Y=EY, TARGET_IDS=TI)
        def fake_fit(Xtr, ytr, Xva, yva, seed, ep=None):
            m = LogisticRegression(max_iter=100, C=0.05).fit(Xtr.reshape(len(Xtr), -1), ytr); vf = float(f1_score(yva, m.predict(Xva.reshape(len(Xva), -1)), average='macro'))
            return dict(model=m, info=dict(val_f1=vf, best_epoch=3, epochs_run=23, stop_reason='early_stop_patience', restore_delta=0.0, history=[vf]))
        g.update(FIT_FN=fake_fit, PREDICT_FN=lambda m, Xe: m.predict(Xe.reshape(len(Xe), -1)))
        cap = make_cap(st, cr); cells, grid = build_plan(cap); assert len(cells) > 0 and {c['target'] for c in cells} >= {'ARG', 'CHN', 'FRA', 'AUS'}
        cells = [c for c in cells if c['k'] <= 2 or c['target'] == 'FRA'][:40]
        with tempfile.TemporaryDirectory() as td:
            csvp = os.path.join(td, 'results.csv'); buf = io.StringIO()
            with contextlib.redirect_stdout(buf): r1 = run_seed(0, cells, csvp, td); r1b = run_seed(1, cells, csvp, td)
            df = load_results(csvp); assert r1 == 'complete' and len(df) == 2 * len(cells) and (df['status'] == 'ok').all(), df['status'].value_counts().to_dict()
            assert len(os.listdir(os.path.join(td, 'runs'))) == 2 * len(cells) and not [f for f in os.listdir(td) if f.startswith('.tmp_')]
            z = np.load(os.path.join(td, 'runs', hashlib.md5(df.run_id.iloc[0].encode()).hexdigest()[:16] + '.npz'))
            assert {'y_true', 'y_pred', 'target_sample_ids', 'usa_train_ids', 'cm7', 'cm_paper_rule', 'paper_f1_per_class', 'train_counts'} <= set(z.files) and len(z['usa_train_ids']) + len(z['usa_val_ids']) == df.budget.iloc[0]
            assert (pd.to_numeric(df.scorer_maxdiff) <= SCORER_TOL).all()
            with contextlib.redirect_stdout(buf): run_seed(0, cells, csvp, td)
            assert len(load_results(csvp)) == 2 * len(cells), 'resume re-ran finished runs'
            kc = df[df.target == 'ARG'].groupby('budget')['key_n'].nunique().max(); assert kc == 1
            cs = cell_stats(df); assert len(cs) > 0; txt = gate_report(df, 0, cells); assert 'GATE REPORT' in txt and 'INTEGRITY' in txt
            # skill failure never aborts:
            g['FIT_FN'] = lambda *a, **k: dict(model=None, info=dict(val_f1=0.1, best_epoch=47, epochs_run=50, stop_reason='max_epochs', restore_delta=0.0, history=[0.1]))
            g['PREDICT_FN'] = lambda m, Xe: np.full(len(Xe), CROP_ID['other'])
            csv2 = os.path.join(td, 'skill.csv')
            with contextlib.redirect_stdout(buf): run_seed(0, cells[:8], csv2, td)
            d2 = load_results(csv2); assert len(d2) == 8 and (d2.status == 'ok').all() and d2.skill_flags.str.contains('WARN_VAL').all()
            # 5 consecutive integrity failures abort:
            g['FIT_FN'] = lambda *a, **k: dict(model=None, info=dict(val_f1=0.9, best_epoch=3, epochs_run=23, stop_reason='x', restore_delta=0.5, history=[0.9]))
            csv3 = os.path.join(td, 'integ.csv')
            try:
                with contextlib.redirect_stdout(buf): run_seed(0, cells[:12], csv3, td)
                raise RuntimeError('did not abort')
            except RuntimeError as e: assert 'consecutive integrity' in str(e), e
            assert len(load_results(csv3)) == MAX_CONSEC_INTEGRITY
        return True
    finally:
        g.update(saved_cfg)
        for n, v in saved.items():
            if isinstance(v, str) and v == '<unset>': g.pop(n, None)
            else: g[n] = v

@_t('END-TO-END dry run with a stand-in model: plan, run, 2 scorers, gates, npz, CSV-last, resume, stats, gate report, skill warnings never abort, 5 integrity failures abort')
def _():
    assert selftest_pipeline()

N_FAIL_A = sum(1 for r in TEST_REPORT if r[1] != 'PASS')
for name, res, msg in TEST_REPORT: print(f'  [{res}] {name}' + (f'\n         -> {msg}' if msg else ''))
TESTS_A_PASSED = (N_FAIL_A == 0)
print('\nTEST GATE PART A:', 'ALL PASSED' if TESTS_A_PASSED else f'{N_FAIL_A} FAILED -- training is BLOCKED')

## Data functions (strict verification against paper v3 Table 2)

In [ ]:
# ======================================================================================
# DATA: load Sentinel-2 median time series (official files), de-duplicate like the authors' loader, attach a US STATE to every USA sample,
#       and VERIFY everything before any training.   (Torch-free: arrays stay numpy; the torch cell converts them.)
# ======================================================================================
ALL_STATES = ['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA','HI','ID','IL','IN','IA','KS','KY','LA','ME','MD','MA','MI','MN','MS','MO','MT',
              'NE','NV','NH','NJ','NM','NY','NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA','WV','WI','WY']

def unique_row_index(a):
    # indices of the FIRST occurrence of every distinct feature row == what the authors' data_load() keeps via np.unique(axis=0, return_index=True)
    a = np.ascontiguousarray(np.asarray(a).reshape(len(a), -1)); seen, keep = set(), []
    for i in range(len(a)):
        h = hashlib.blake2b((a[i] + 0.0).tobytes(), digest_size=16).digest()   # +0.0 turns -0.0 into 0.0 (np.unique treats them as equal)
        if h not in seen: seen.add(h); keep.append(i)
    return np.asarray(keep, dtype=np.int64)

def find_npz(data_dir, country):
    name = f'median_S2_{country}_month1-12_day5.npz'; hits = []
    for root, _d, files in os.walk(data_dir):
        if name in files: hits.append(os.path.join(root, name))
    _check(hits, f'{name} not found under {data_dir}')
    hits.sort(key=lambda p: ('for_CropNet' not in p, len(p)))
    return hits[0]

def load_country(path, country):
    z = np.load(path); lbl = z['lbl'].astype(int)
    cand = ['fea'] if 'fea' in z.files else [k for k in z.files if k != 'lbl' and z[k].ndim >= 2 and z[k].shape[0] == len(lbl)]
    _check(cand, f'{country}: no per-sample feature array (keys={z.files})')
    a = z[cand[0]].astype(np.float32); band_axes = [ax for ax in range(1, a.ndim) if a.shape[ax] == 10]
    _check(band_axes, f'{country}: cannot find the 10-band axis in shape {a.shape}')
    a = np.moveaxis(a, band_axes[0], 1).reshape(a.shape[0], 10, -1)
    _check(len(lbl) == len(a), f'{country}: label/feature length mismatch')
    _check(lbl.min() >= 0 and lbl.max() <= 6, f'{country}: label ids outside 0..6')
    nan = int((~np.isfinite(a)).sum()); _check(nan == 0, f'{country}: {nan} non-finite feature values (NaN/inf) -- CropNet would produce NaN')
    return lbl, a

# ---------------------------------------------------------------- USA state assignment (alignment logic reused from the audited Phase 3 notebook)
TOKENS_LONGEST_FIRST = sorted(RAW_TO_CROP, key=len, reverse=True)
def parse_usa_filename(fname):
    m = re.match(r'^S2_USA_(.+)\.csv$', fname)
    if not m: return None
    rest = m.group(1)
    for tok in TOKENS_LONGEST_FIRST:
        if rest == tok or rest.startswith(tok + '_'):
            tail = rest[len(tok) + 1:] if rest != tok else ''
            return tok, (tail.upper() if len(tail) == 2 else None)
    return None

def read_geo_column(path):
    df = pd.read_csv(path, usecols=['.geo']); lons = np.empty(len(df)); lats = np.empty(len(df))
    for i, g in enumerate(df['.geo'].values):
        c = json.loads(g)['coordinates']; lons[i] = float(c[0]); lats[i] = float(c[1])
    return lons, lats

def between_state_ratio(F, crop_ids, state_codes, rng, n_perm=8):
    def one(X, codes):
        if len(np.unique(codes)) < 2: return float('nan')
        order = np.argsort(codes, kind='stable'); Xs, gs = X[order], codes[order]
        starts = np.r_[0, np.flatnonzero(np.diff(gs)) + 1]; cnt = np.diff(np.r_[starts, len(gs)])
        means = np.add.reduceat(Xs, starts, axis=0) / cnt[:, None]; gm = X.mean(0)
        return float((cnt[:, None] * (means - gm) ** 2).sum() / ((X - gm) ** 2).sum())
    lifts = []
    for c in np.unique(crop_ids):
        m = np.flatnonzero(crop_ids == c); X, g = F[m], state_codes[m]; obs = one(X, g)
        perm = float(np.nanmean([one(X, g[rng.permutation(len(g))]) for _ in range(n_perm)]))
        if perm > 0: lifts.append(obs / perm)
    return float(np.mean(lifts)) if lifts else float('nan')

def build_usa_meta(geo_by_file, usa_lbl_raw, F, sub, keep_usa, assign_states_fn, log=print):
    file_rows, lons, lats, fidx = [], [], [], []
    for fname in sorted(geo_by_file):
        parsed = parse_usa_filename(fname)
        if parsed is None: continue
        tok, st_fn = parsed; lo, la = geo_by_file[fname]; fi = len(file_rows)
        file_rows.append(dict(file_idx=fi, fname=fname, raw_tok=tok, crop=RAW_TO_CROP[tok], crop_id=CROP_ID[RAW_TO_CROP[tok]], state_fn=st_fn, n_rows=len(lo)))
        lons.append(np.asarray(lo)); lats.append(np.asarray(la)); fidx.append(np.full(len(lo), fi))
    _check(file_rows, 'no S2_USA_*.csv geometry found')
    file_table = pd.DataFrame(file_rows)
    geo = pd.DataFrame({'file_idx': np.concatenate(fidx), 'lat': np.concatenate(lats), 'lon': np.concatenate(lons)})
    if len(geo) != len(usa_lbl_raw):
        raise RuntimeError(f'raw USA points {len(geo)} (from {len(file_table)} files) != rows in the USA NPZ {len(usa_lbl_raw)}. '
                           'The raw USA CSV set is incomplete (typically a truncated Google-Drive listing). See the platform cell for how to fix it.')
    ORDERINGS = {
        'crop_id,token,state':      lambda ft: ft.sort_values(['crop_id', 'raw_tok', 'state_fn']),
        'crop_id,token_desc,state': lambda ft: ft.sort_values(['crop_id', 'raw_tok', 'state_fn'], ascending=[True, False, True]),
        'crop_id,state,token':      lambda ft: ft.sort_values(['crop_id', 'state_fn', 'raw_tok']),
        'filename':                 lambda ft: ft.sort_values('fname'),
    }
    offsets = np.r_[0, np.cumsum(file_table.sort_values('file_idx')['n_rows'].values)]
    def build_row_order(fo): return np.concatenate([np.arange(offsets[fi], offsets[fi] + n) for fi, n in zip(fo['file_idx'], fo['n_rows'])])
    candidates = {}
    for name, fn in ORDERINGS.items():
        fo = fn(file_table); pred = np.repeat(fo['crop_id'].values, fo['n_rows'].values)
        if (pred == usa_lbl_raw).all(): candidates[name] = build_row_order(fo); log(f'  ordering {name:28s} exact label-sequence match: True')
    _check(candidates, 'No file ordering reproduces the NPZ label sequence -- state alignment impossible, stopping.')
    uniq = {}
    for name, ro in candidates.items():                                   # identical row orders are ONE hypothesis (keep the first name)
        if not any(np.array_equal(ro, u) for u in uniq.values()): uniq[name] = ro
    candidates = uniq
    geo['state'] = assign_states_fn(geo['lat'].values, geo['lon'].values)
    _check(pd.notna(geo['state']).all(), 'some USA points got no state')
    state_code_of = {s: i for i, s in enumerate(ALL_STATES)}; rng = np.random.default_rng(0)
    best_name, best_lift, best_order = None, -1, None
    lifts = {}
    for name, ro in candidates.items():
        st = np.array([state_code_of[s] for s in geo['state'].values[ro]])
        lifts[name] = between_state_ratio(F, usa_lbl_raw[sub], st[sub], rng); log(f'  ordering {name:28s} state-alignment lift: {lifts[name]:.2f}x')
    top = max(lifts.values())
    # near-tie (within 3%, i.e. permutation noise): keep the FIRST ordering in the audited list (the one validated in the Phase 3 notebook)
    best_name = next(n for n in ORDERINGS if n in lifts and lifts[n] >= 0.97 * top); best_lift, best_order = lifts[best_name], candidates[best_name]
    _check(best_lift >= 2.0, f'Best ordering "{best_name}" lift {best_lift:.2f} (<2): state alignment unsupported, stopping.')
    log(f'  -> aligned ordering "{best_name}" (lift {best_lift:.1f}x)')
    # ambiguity guard: another ordering that assigns DIFFERENT states to some rows but scores almost as well is a real risk
    for name, ro in candidates.items():
        if name == best_name: continue
        differ = float((geo['state'].values[ro] != geo['state'].values[best_order]).mean())
        lift_o = between_state_ratio(F, usa_lbl_raw[sub], np.array([state_code_of[x] for x in geo['state'].values[ro]])[sub], np.random.default_rng(1))
        if differ > 0 and lift_o > best_lift / 1.15:
            log(f'  !! WARNING: ordering "{name}" differs on {100 * differ:.1f}% of the rows but its lift ({lift_o:.1f}x) is within 15% of the best -- state labels of those rows are uncertain')
    full = pd.DataFrame({'npz_row': np.arange(len(usa_lbl_raw)), 'crop_id': usa_lbl_raw, 'state': geo['state'].values[best_order]})
    _check((full['crop_id'].values == usa_lbl_raw).all(), 'alignment broke the label order')
    meta = full.iloc[keep_usa].reset_index(drop=True)
    meta['crop'] = meta['crop_id'].map(CROP_LABELS)
    return meta, best_name, best_lift

def assign_states_census(lat, lon, cache_dir):
    # point-in-polygon on the US Census 2018 state polygons; nearest state as the fallback for coastal points
    import geopandas as gpd, urllib.request
    url = 'https://www2.census.gov/geo/tiger/GENZ2018/shp/cb_2018_us_state_20m.zip'; zp = os.path.join(cache_dir, 'cb_2018_us_state_20m.zip')
    if not os.path.exists(zp):
        last = None
        for _ in range(4):
            try: urllib.request.urlretrieve(url, zp); break
            except Exception as e: last = e; time.sleep(3)
        _check(os.path.exists(zp), f'could not download the Census state polygons ({last}); check that Internet is ON')
    try: sg = gpd.read_file(zp)
    except Exception: sg = gpd.read_file('zip://' + zp)
    sg = sg[sg['STUSPS'].isin(ALL_STATES)][['STUSPS', 'geometry']].to_crs('EPSG:4326')
    pts = gpd.GeoDataFrame({'k': np.arange(len(lat))}, geometry=gpd.points_from_xy(lon, lat), crs='EPSG:4326')
    j = gpd.sjoin(pts, sg, how='left', predicate='within'); j = j[~j.index.duplicated(keep='first')]
    state = j['STUSPS'].reindex(pts.index).values.astype(object)
    un = pd.isna(state)
    if un.any():
        near = gpd.sjoin_nearest(pts[un].to_crs('EPSG:5070'), sg.to_crs('EPSG:5070'), how='left', distance_col='d')
        near = near[~near.index.duplicated(keep='first')]; state[un] = near['STUSPS'].reindex(pts.index[un]).values
    return state

def load_all_data(data_dir, out_dir, get_geo_fn, extra_meta_paths=(), assign_fn=None):
    cache_dir = os.path.join(out_dir, 'cache'); os.makedirs(cache_dir, exist_ok=True)
    needed = ['USA'] + [t for t in TARGETS_TO_RUN]; FEATURES, NPZ_LBL, INFO, F_lift, sub = {}, {}, {}, None, None
    keep_usa, usa_lbl_raw = None, None
    for c in needed:
        path = find_npz(data_dir, c); lbl, a = load_country(path, c); keep = unique_row_index(a)
        if c == 'USA':
            usa_lbl_raw, keep_usa = lbl, keep
            sub = np.random.default_rng(0).choice(len(lbl), size=min(30000, len(lbl)), replace=False)
            F = a.reshape(len(a), -1)[sub].astype(np.float32); F_lift = (F - F.mean(0)) / (F.std(0) + 1e-6)
        FEATURES[c] = np.ascontiguousarray(a[keep][:, None, :, :]); NPZ_LBL[c] = lbl[keep]
        INFO[c] = dict(path=path, raw=len(lbl), kept=len(keep), dup=len(lbl) - len(keep), shape=tuple(FEATURES[c].shape), keep=keep, raw_by_class=pd.Series(lbl).map(CROP_LABELS).value_counts().reindex(CROP_ORDER, fill_value=0).to_dict(), mean=float(a.mean()), std=float(a.std()),
                       vmin=float(a.min()), vmax=float(a.max())); del a
    # --- USA state table: cache -> attached meta -> build from raw geometry
    meta, how = None, ''
    for p in [os.path.join(cache_dir, 'usa_meta.csv')] + list(extra_meta_paths):
        if p and os.path.exists(p):
            m = pd.read_csv(p)
            if len(m) == len(NPZ_LBL['USA']) and (m['crop_id'].values == NPZ_LBL['USA']).all() and m['state'].notna().all():
                m['crop'] = m['crop_id'].map(CROP_LABELS); meta, how = m[['npz_row', 'crop_id', 'state', 'crop']], f'cached state table: {p}'; break
            print(f'  (ignoring {p}: does not match the USA NPZ)')
    if meta is None:
        meta, name, lift = build_usa_meta(get_geo_fn(), usa_lbl_raw, F_lift, sub, keep_usa, assign_fn or (lambda lat, lon: assign_states_census(lat, lon, cache_dir)))
        meta[['npz_row', 'crop_id', 'state']].to_csv(os.path.join(cache_dir, 'usa_meta.csv'), index=False); how = f'built from raw geometry (ordering {name}, lift {lift:.1f}x)'
    print('USA state table:', how)
    return FEATURES, NPZ_LBL, INFO, meta

def row_hash_set(F):
    a = np.ascontiguousarray(F.reshape(len(F), -1)); return {hashlib.blake2b((a[i] + 0.0).tobytes(), digest_size=16).digest() for i in range(len(a))}

def verify_data(FEATURES, NPZ_LBL, INFO, meta):
    # EVERY check here is a STOP (spec section 1): raw counts, label sets, class counts vs paper v3 Table 2, no target row inside the USA table, state alignment.
    print('\n================ DATA VERIFICATION (paper arXiv 2509.03497 v3, Table 2) ================')
    rows = []
    for c in INFO:
        rows.append(dict(country=c, raw=INFO[c]['raw'], table2_total=TABLE2_TOTAL[c], kept_after_dedup=INFO[c]['kept'], removed_duplicates=INFO[c]['dup'], shape=INFO[c]['shape'], **{f'kept_{k}': v for k, v in
                         pd.Series(NPZ_LBL[c]).map(CROP_LABELS).value_counts().reindex(CROP_ORDER, fill_value=0).to_dict().items()}))
    print(pd.DataFrame(rows).to_string(index=False))
    problems = []
    for c in INFO:                                                                  # raw row counts == Table 2 totals; raw class counts == Table 2 class counts
        if INFO[c]['raw'] != TABLE2_TOTAL[c]: problems.append(f'{c}: raw rows {INFO[c]["raw"]} != Table 2 total {TABLE2_TOTAL[c]}')
        if INFO[c]['raw_by_class'] != TABLE2[c]: problems.append(f'{c}: raw class counts {INFO[c]["raw_by_class"]} != Table 2 {TABLE2[c]}')
    for t in TARGETS_TO_RUN:                                                        # label set per target == Table 2 (raw AND after de-duplication)
        got = sorted([CROP_LABELS[i] for i in np.unique(NPZ_LBL[t])], key=CROP_ORDER.index); exp = sorted(DEFINED_CLASSES[t], key=CROP_ORDER.index)
        rawset = sorted([c for c, n in INFO[t]['raw_by_class'].items() if n > 0], key=CROP_ORDER.index)
        print(f'  label set {t}: after dedup {got} | raw {rawset} | Table 2 {exp} -> {"OK" if got == exp and rawset == exp else "MISMATCH"}')
        if got != exp or rawset != exp: problems.append(f'{t}: label set data(raw {rawset}, dedup {got}) != Table 2 {exp}')
    for c in INFO:
        if INFO[c]['shape'][1:3] != (1, 10): problems.append(f'{c}: unexpected tensor shape {INFO[c]["shape"]} (need (N,1,10,T))')
        elif INFO[c]['shape'][3] < 67: problems.append(f'{c}: time axis {INFO[c]["shape"][3]} too short for the official DOY 121-334 window (needs >= 67 five-day steps)')
    if problems:
        raise RuntimeError('STOP -- data conflicts with the paper (Table 2). Report, do not resolve silently:\n  - ' + '\n  - '.join(problems))
    print('\nde-duplication difference (raw -> kept):  ' + ' | '.join(f"{c}: -{INFO[c]['dup']}" for c in INFO))
    print('feature scale per country (mean / std / min / max):')
    for c in INFO: print(f"  {c}: {INFO[c]['mean']:.1f} / {INFO[c]['std']:.1f} / {INFO[c]['vmin']:.1f} / {INFO[c]['vmax']:.1f}")
    means = [INFO[c]['mean'] for c in INFO]
    if max(means) > 10 * max(min(means), 1e-9): print('  !! WARNING: country feature means differ by >10x -- check that all files use the same reflectance scale')
    _check(len(meta) == len(FEATURES['USA']) and (meta['crop_id'].values == NPZ_LBL['USA']).all(), 'USA state table is not aligned with the USA features')
    _check(set(meta['state']) <= set(ALL_STATES), 'unknown state codes in the USA table')
    # LEAKAGE: no target sample may be identical to any USA row (targets never enter training)
    hu = row_hash_set(FEATURES['USA'])
    for t in TARGETS_TO_RUN:
        n_dup = len(hu & row_hash_set(FEATURES[t])); print(f'  leakage check USA vs {t}: {n_dup} identical rows')
        _check(n_dup == 0, f'{t}: {n_dup} target rows are identical to USA rows -- leakage; STOP and report')
    big = meta.groupby('state').size(); print(f"USA: {len(meta)} samples, {meta['state'].nunique()} states ({int((big >= MIN_STATE_ROWS).sum())} with >= {MIN_STATE_ROWS} rows) -- state table aligned row by row")
    print('DATA VERIFICATION PASSED\n')

## Acquire data and code (authors' repo pinned to the spec commit; namespace guard)

In [ ]:
# ======================================================================================
# [COLAB] DATA + CODE: use the files already in Drive; download only what is missing.  The authors' repo is PINNED to the commit in the spec.
# ======================================================================================
NEEDED = ['USA'] + list(TARGETS_TO_RUN)
def _have(c): return bool(glob.glob(os.path.join(DATA_DIR, '**', f'median_S2_{c}_month1-12_day5.npz'), recursive=True))
if not all(_have(c) for c in NEEDED):
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='x-ytong/CropGlobe', repo_type='dataset', local_dir=DATA_DIR)
assert all(_have(c) for c in NEEDED), f'missing Sentinel-2 median files for {[c for c in NEEDED if not _have(c)]} in {DATA_DIR}'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/x-ytong/CropNet.git', REPO_DIR], check=True)
def _git(*a): return subprocess.run(['git', '-C', REPO_DIR, *a], capture_output=True, text=True)
if _git('rev-parse', 'HEAD').stdout.strip() != EXPECTED_CROPNET_COMMIT:
    _git('fetch', '--all', '--tags')
    if _git('cat-file', '-e', EXPECTED_CROPNET_COMMIT + '^{commit}').returncode != 0:
        raise RuntimeError(f'STOP -- spec conflict: commit {EXPECTED_CROPNET_COMMIT} does not exist in x-ytong/CropNet. Report; do not substitute another commit silently.')
    r = _git('checkout', '--detach', EXPECTED_CROPNET_COMMIT); assert r.returncode == 0, r.stderr
GIT_COMMIT = _git('rev-parse', 'HEAD').stdout.strip(); assert GIT_COMMIT == EXPECTED_CROPNET_COMMIT, f'repo at {GIT_COMMIT}, spec says {EXPECTED_CROPNET_COMMIT}'
_dirty = _git('status', '--porcelain', '--untracked-files=no').stdout.strip()
assert not _dirty, f'the authors\' code must be imported UNCHANGED, but the repo has local modifications:\n{_dirty}'
assert os.path.isfile(os.path.join(REPO_DIR, 'models', 'CropNet.py')), 'authors\' repo is incomplete'
print('authors\' repo pinned at', GIT_COMMIT, '(clean, unmodified)')

def get_usa_geo_by_file():
    cache = os.path.join(OUT_DIR, 'cache', 'usa_geo.pkl'); os.makedirs(os.path.dirname(cache), exist_ok=True)
    if os.path.exists(cache): return pickle.load(open(cache, 'rb'))
    assert os.path.isdir(RAW_USA_DIR), f'{RAW_USA_DIR} not found'
    paths = sorted(glob.glob(os.path.join(RAW_USA_DIR, '**', '*.csv'), recursive=True)); geo = {}
    for i, p in enumerate(paths):
        fn = os.path.basename(p)
        if parse_usa_filename(fn) is None: continue
        assert fn not in geo, f'duplicate file name {fn}'
        geo[fn] = read_geo_column(p)
        if (i + 1) % 25 == 0: print(f'  read geometry of {i + 1}/{len(paths)} raw USA files')
    pickle.dump(geo, open(cache, 'wb')); return geo

# ---- namespace guard: never mix with, or overwrite, results of an earlier notebook (spec 7)
def check_namespace(write=False):
    mp, rp = os.path.join(OUT_DIR, 'manifest.json'), os.path.join(OUT_DIR, 'results.csv')
    snap = {k: json.loads(json.dumps(globals()[k], default=str)) for k in LOCKED}
    if os.path.exists(mp):
        m = json.load(open(mp))
        if m.get('spec') != SPEC_TAG:
            raise RuntimeError(f'STOP: {OUT_DIR} holds a manifest from an EARLIER notebook (spec={m.get("spec")}). Spec 7 says never touch earlier outputs: rename/move that folder (keep it), then re-run.')
        diff = [k for k in LOCKED if m['locked'].get(k) != snap[k]]
        if diff: raise RuntimeError(f'STOP: locked design differs from the manifest in {diff}. Use a new namespace instead of mixing designs.')
    elif os.path.exists(rp):
        raise RuntimeError(f'STOP: {rp} exists without a v3-spec manifest (results of an earlier notebook). Move that folder aside (do not delete) and re-run.')
    elif write:
        man = dict(spec=SPEC_TAG, experiment=EXPERIMENT_VERSION, created=datetime.datetime.now().isoformat(timespec='seconds'), locked=snap, cropnet_commit=GIT_COMMIT,
                   table2=TABLE2, table3_reference_context_only=TABLE3_REF_MF1)
        atomic_write(mp, lambda f: f.write(json.dumps(man, indent=1)), mode='w')
    return True
check_namespace(write=False)
print('Colab data + code ready. Sentinel-2 files for', NEEDED, 'found. Namespace OK:', OUT_DIR)

## Test gate - Part B (our scorer vs the authors' own `remap_prediction`)

In [ ]:
# ======================================================================================
# PRE-TRAINING TEST GATE, PART B: our scorer vs the AUTHORS' OWN util/label.py::remap_prediction (needs the pinned repo from the cell above)
# ======================================================================================
import inspect, importlib
for _m in ('util', 'util.label'):
    if _m in sys.modules and REPO_DIR not in str(getattr(sys.modules[_m], '__file__', '') or ''): del sys.modules[_m]
if REPO_DIR in sys.path: sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)
UL = importlib.import_module('util.label'); assert os.path.abspath(inspect.getfile(UL)).startswith(os.path.abspath(REPO_DIR))
AUTH_REMAP = UL.remap_prediction
print('authors\' remap_prediction signature:', inspect.signature(AUTH_REMAP)); print(inspect.getsource(AUTH_REMAP))

def call_authors_remap(y_pred, target):
    # we do not know the exact signature in advance: map parameters by name, and try the usual spellings of the target name; the result must be an array of class ids
    params = list(inspect.signature(AUTH_REMAP).parameters)
    def build(tv):
        args = []
        for p in params:
            pl = p.lower()
            if any(s in pl for s in ('pred', 'label', 'arr', 'out')) or pl in ('y', 'yp', 'x'): args.append(np.array(y_pred, copy=True))
            elif any(s in pl for s in ('target', 'country', 'region', 'name', 'dataset', 'data')): args.append(tv)
            else: raise TypeError(f'cannot map parameter "{p}" of remap_prediction')
        return args
    last = None
    for tv in (target, target.lower(), {'ARG': 'Argentina', 'CHN': 'China', 'AUS': 'Australia', 'FRA': 'France'}[target]):
        try:
            out = AUTH_REMAP(*build(tv)); out = out.numpy() if hasattr(out, 'numpy') else np.asarray(out)
            if out.shape == np.asarray(y_pred).shape: return out.astype(int), tv
        except Exception as e: last = e
    raise RuntimeError(f'could not call the authors\' remap_prediction for {target}: {last!r}')

TESTS_B_PASSED = False
_rng = np.random.default_rng(7); _bad = []
for _t_ in ['ARG', 'CHN', 'AUS', 'FRA']:
    yp = _rng.integers(0, 7, 5000); ref, used = call_authors_remap(yp, _t_)
    ours = remap_paper_rule(yp, _t_)
    if not (ref == ours).all(): _bad.append(f'{_t_}: authors vs ours differ on {int((ref != ours).sum())}/5000 predictions (call variant {used!r}); mapping authors {dict(zip(*np.unique(ref[ref != yp], return_counts=True)))}')
    yt = _rng.choice([CROP_ID[c] for c in DEFINED_CLASSES[_t_]], 5000)
    f_auth = f1_score(yt, ref, average='macro', labels=np.unique(yt), zero_division=0); sc = score_run(_t_, yt, yp, np.array([100] * 7))
    if abs(sc['paper_f1'] - f_auth) > 1e-12: _bad.append(f'{_t_}: scorer paper-rule F1 {sc["paper_f1"]} != F1 on the authors\' remap {f_auth}')
    print(f'  [{"FAIL" if any(b.startswith(_t_) for b in _bad) else "PASS"}] {_t_}: remap identical to authors\' remap_prediction on 5000 random predictions; scorer == f1_score(y_true, authors_remap, labels=np.unique(y_true)) (variant {used!r})')
if _bad: raise RuntimeError('STOP -- the paper rule in the spec does not match the authors\' released util/label.py. Report, do not resolve silently:\n  - ' + '\n  - '.join(_bad))
TESTS_B_PASSED = True
TESTS_PASSED = bool(TESTS_A_PASSED and TESTS_B_PASSED)
print('\nTEST GATE: Part A', 'PASS' if TESTS_A_PASSED else 'FAIL', '| Part B', 'PASS' if TESTS_B_PASSED else 'FAIL', '->', 'training allowed after the plan approval' if TESTS_PASSED else 'TRAINING BLOCKED')

## Load and verify the data (every mismatch with Table 2 is a STOP)

In [ ]:
# ======================================================================================
# LOAD + VERIFY THE DATA  (nothing is trained here)
# ======================================================================================
FEATURES, NPZ_LBL, INFO, meta = load_all_data(DATA_DIR, OUT_DIR, get_usa_geo_by_file, extra_meta_paths=EXTRA_META_PATHS)
verify_data(FEATURES, NPZ_LBL, INFO, meta)

## Plan table - read it, then set `APPROVED['plan'] = True`

In [ ]:
# ======================================================================================
# PLAN TABLE (nothing is trained here).  READ IT, then set APPROVED['plan'] = True in the config cell.
# ======================================================================================
assert TESTS_PASSED, 'the test gate (Part A + Part B) must pass first'
USA_STATE   = meta['state'].values.astype(str); USA_CROP = meta['crop'].values.astype(str); USA_CROP_ID = meta['crop_id'].values.astype(int)
USA_NPZ_ROW = meta['npz_row'].values.astype(np.int64)                                   # row index in the ORIGINAL (pre-dedup) USA npz -> saved as sample ids
TARGET_IDS  = {t: INFO[t]['keep'] for t in TARGETS_TO_RUN}                              # original npz row of every kept target sample
POOLS = build_pools(USA_STATE, USA_CROP); CAP = make_cap(USA_STATE, USA_CROP)
os.makedirs(os.path.join(OUT_DIR, 'tables'), exist_ok=True)
CAP.sort_values('total', ascending=False).to_csv(os.path.join(OUT_DIR, 'tables', 'state_crop_table.csv'))
print(f'USA states: {len(CAP)} | states with >= {MIN_STATE_ROWS} rows: {int((CAP.total >= MIN_STATE_ROWS).sum())}')
print(CAP.sort_values('total', ascending=False).head(25).to_string())

PLAN_CELLS, PLAN_GRID = build_plan(CAP)
PLAN_CELLS_RUN = [c for c in PLAN_CELLS if not c['pending']]
PLAN_GRID.to_csv(os.path.join(OUT_DIR, 'tables', 'feasibility_grid.csv'), index=False)
pd.DataFrame([dict(c, states='-'.join(c['states'])) for c in PLAN_CELLS]).to_csv(os.path.join(OUT_DIR, 'tables', 'plan_cells.csv'), index=False)

print('\n================ PLAN TABLE ================')
print(f'budgets {BUDGETS} | k {K_LIST} | seeds {SEEDS} | aug {AUG} | max {MAX_COMBOS} combos for k>=2 (k=1 census) | infeasible if <{MIN_COMBOS} combos or <{MIN_KPOINTS} k-points')
print('key-crop shares (fixed at every k; the rest sampled naturally):', {t: effective_key_shares(t) if t != 'AUS' else 'NOT SPECIFIED' for t in TARGETS_TO_RUN})
for t in TARGETS_TO_RUN:
    print(f'\n--- {t} ({TARGET_ROLE[t]}; primary view: {PRIMARY_VIEW[t][0]}{"_" + PRIMARY_VIEW[t][1] if PRIMARY_VIEW[t][1] else " (paper rule)"}; K_MAX={K_MAX[t]}) ---')
    g = PLAN_GRID[PLAN_GRID.target == t]
    for B in BUDGETS:
        gb = g[g.budget == B]; kc = key_counts(t, B)
        print(f'  B={B}: key counts {kc if kc else "none (fully natural)"} + natural rest {B - sum(kc.values())} | pool = {int(gb.pool_size.iloc[0])} state(s): {gb.pool.iloc[0]}')
        print('      ' + ' | '.join(f'k={int(r.k)}: ' + (f'{int(r.n_combos)} combo(s) of {int(r.n_total_combos)}' + (' [census]' if r.k == 1 else '') if r.feasible else f'INFEASIBLE ({r.note.strip()})') for r in gb.itertuples()))
    n_runs = sum(1 for c in PLAN_CELLS_RUN if c['target'] == t) * len(SEEDS)
    print(f'  -> {sum(1 for c in PLAN_CELLS if c["target"] == t)} cells x {len(SEEDS)} seeds = {sum(1 for c in PLAN_CELLS if c["target"] == t) * len(SEEDS)} runs' + ('   *** EXCLUDED UNTIL YOU CONFIRM AUS (see below) ***' if t == 'AUS' and not AUS_SPEC_CONFIRMED else f'   ({n_runs} will run)'))
if 'AUS' in TARGETS_TO_RUN and not AUS_SPEC_CONFIRMED:
    print('\n!! AUS DECISION NEEDED: the spec says "no sweep, census of {LA, FL}, k<=2" but gives NO key-crop share, so the budget composition is undefined.'
          '\n   AUS is shown above with fully natural sampling for feasibility only and is EXCLUDED from the run.  Set AUS_KEY_SHARES_IF_CONFIRMED (e.g. {"sugarcane": 0.20}) and AUS_SPEC_CONFIRMED=True to include it.')
print(f'\nTOTAL training runs that will execute: {len(PLAN_CELLS_RUN) * len(SEEDS)}  ({len(PLAN_CELLS_RUN)} cells x {len(SEEDS)} seeds); GPU time is estimated after the smoke test.')
_check(len(PLAN_CELLS_RUN) > 0, 'EMPTY PLAN: no cell is feasible -- see the tables above')
print('\n>>> APPROVAL REQUIRED: if this plan is right, set APPROVED["plan"] = True in the config cell and continue.  Nothing has been trained.')

## Torch backend (authors' CropNet) + 2-epoch smoke test

In [ ]:
# ======================================================================================
# TORCH BACKEND -- the AUTHORS' CropNet and CropGlobeDataset (unchanged, pinned commit), official recipe:
#   CropNet(data_channel=1, classes=7), ALL 7 outputs trained | Adam lr 1e-4, weight_decay 0, no scheduler | batch 256
#   CrossEntropyLoss with balanced class weights computed on the drawn sample (train+val), zero-sample crops -> weight 0
#   official DOY 121-334 window (aug_mode='none') for train, validation AND test | augmentation OFF
#   MAX 50 epochs, early stopping on VALIDATION macro-F1 (patience 15, min 20 epochs, strictly greater), best weights restored and re-verified
# ======================================================================================
assert TESTS_PASSED, 'test gate (Part A + Part B) has not passed -- no GPU time'
assert APPROVED['plan'], 'the plan table has not been approved: set APPROVED["plan"] = True in the config cell after reading it'
import torch
from torch.utils.data import TensorDataset, DataLoader
os.environ.setdefault('TQDM_DISABLE', '1')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE, '|', torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'NO GPU (very slow) -- enable a GPU accelerator')
for _m in ('models', 'data'):
    if _m in sys.modules and REPO_DIR not in str(getattr(sys.modules[_m], '__file__', '') or ''): del sys.modules[_m]
if REPO_DIR in sys.path: sys.path.remove(REPO_DIR)
sys.path.insert(0, REPO_DIR)
from models.CropNet import CropNet
from data.augment import CropGlobeDataset
for _cls in (CropNet, CropGlobeDataset): assert os.path.abspath(inspect.getfile(_cls)).startswith(os.path.abspath(REPO_DIR)), f'{_cls} is not the authors\' class'
assert subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip() == EXPECTED_CROPNET_COMMIT
_probe = CropGlobeDataset()
assert (_probe.dayItv, _probe.staDOY, _probe.endDOY, _probe.target_len) == (5, 121, 334, 43), 'unexpected official DOY window'
print(f'official DOY window: day {_probe.staDOY}-{_probe.endDOY}, step {_probe.dayItv} -> {_probe.target_len} steps; model input = 10 bands x {_probe.target_len}')
assert AUG == 'off'

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def official_window(X, y):
    ds = CropGlobeDataset(aug_mode='none')(TensorDataset(torch.as_tensor(X, dtype=torch.float32), torch.as_tensor(y).long()))
    return ds.tensors

@torch.no_grad()
def predict_tensor(model, X, bs=4096):
    model.eval(); out = []; X = torch.as_tensor(X) if not torch.is_tensor(X) else X
    for i in range(0, len(X), bs): out.append(model(X[i:i + bs].to(DEVICE, dtype=torch.float32)).argmax(1).cpu())
    return torch.cat(out).numpy()

def fit_model(Xtr, ytr, Xva, yva, seed, epochs_override=None):
    max_ep = int(epochs_override or MAX_EPOCHS); es = EarlyStopper(max_epochs=max_ep)            # hard cap: EarlyStopper refuses > 50
    set_seed(seed)
    ytr_np, yva_np = np.asarray(ytr).astype(int), np.asarray(yva).astype(int)
    w = balanced_class_weights(np.concatenate([ytr_np, yva_np]), NUM_CLASSES)                      # weights from the DRAWN sample (train+val); zero-sample crops -> 0
    train_ds = CropGlobeDataset(aug_mode='none')(TensorDataset(torch.as_tensor(Xtr, dtype=torch.float32), torch.as_tensor(ytr_np).long()))
    val_X, val_y = official_window(Xva, yva_np); val_X = val_X.to(DEVICE); val_y_np = val_y.numpy()
    g = torch.Generator(); g.manual_seed(seed)
    loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, generator=g)
    set_seed(seed)
    model = CropNet(data_channel=1, classes=NUM_CLASSES).to(DEVICE)                               # authors' architecture, 7-class head
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)                  # no scheduler
    crit = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=DEVICE))
    def val_f1(m): return float(f1_score(val_y_np, predict_tensor(m, val_X), average='macro', zero_division=0))
    best_state, hist = None, []
    for ep in range(max_ep):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE); opt.zero_grad(set_to_none=True); loss = crit(model(xb), yb); loss.backward(); opt.step()
        if not math.isfinite(float(loss)): raise FloatingPointError(f'training loss became non-finite at epoch {ep + 1}')
        f = val_f1(model); hist.append(f); improved, stop = es.step(f)
        if improved: best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        if stop: break
    model.load_state_dict(best_state); delta = abs(val_f1(model) - es.best)                         # restored model must reproduce its best validation score
    return dict(model=model, info=dict(val_f1=es.best, best_epoch=es.best_epoch, epochs_run=es.epoch, stop_reason=es.reason, restore_delta=delta, history=hist))

def prepare_eval_sets():
    EX, EY = {}, {}
    for t in TARGETS_TO_RUN:
        y = np.asarray(NPZ_LBL[t]).astype(int); Xw, _ = official_window(FEATURES[t], y); EX[t] = Xw.to(DEVICE); EY[t] = y
        print(f'  eval set {t}: {len(y)} samples (full, de-duplicated), window tensor {tuple(Xw.shape)}')
    return EX, EY

# ---------------------------------------------------------------- 2-EPOCH SMOKE TEST with the REAL model (spec 9: before any sweep)
print('\nSMOKE TEST'); SMOKE_OK = False
m = CropNet(data_channel=1, classes=NUM_CLASSES).to(DEVICE); out = m(torch.randn(8, 1, 10, 43, device=DEVICE)); assert out.shape == (8, 7); out.sum().backward()
print('  [PASS] official CropNet forward/backward (8,1,10,43) ->', tuple(out.shape))
wx, _ = official_window(np.random.rand(6, 1, 10, 73).astype(np.float32), np.zeros(6, dtype=int)); assert tuple(wx.shape) == (6, 1, 10, 43); print('  [PASS] official window (6,1,10,73) -> (6,1,10,43)')
_r = np.random.default_rng(0); _mu = _r.normal(0, 1, (3, 730)).astype(np.float32)
def _fake(n, labs): y = np.repeat(labs, n); return (_mu[y] + _r.normal(0, 1, (len(y), 730))).astype(np.float32).reshape(len(y), 1, 10, 73), y
Xa, ya = _fake(80, [0, 1, 2]); Xb, yb_ = _fake(20, [0, 1, 2]); t0 = time.time(); r = fit_model(Xa, ya, Xb, yb_, 0, epochs_override=3)
assert set(r['info']) >= {'val_f1', 'best_epoch', 'epochs_run', 'stop_reason', 'restore_delta', 'history'} and r['info']['restore_delta'] <= RESTORE_TOL and math.isfinite(r['info']['val_f1']) and r['info']['epochs_run'] == 3 and r['info']['stop_reason'] == 'max_epochs'
print(f"  [PASS] fit_model (3 epochs; 4 crops absent -> weight 0): val F1 {r['info']['val_f1']:.2f}, best epoch {r['info']['best_epoch']}, restore delta {r['info']['restore_delta']:.1e}, {time.time() - t0:.1f}s")
EVAL_X, EVAL_Y = prepare_eval_sets(); FIT_FN, PREDICT_FN = fit_model, predict_tensor
print('\n  one REAL cell per target, 2 epochs (full pipeline incl. both scorers + integrity checks; results NOT kept):')
for t in TARGETS_TO_RUN:
    cl = [c for c in PLAN_CELLS_RUN if c['target'] == t]
    if not cl: print(f'   {t}: no runnable cell -- skipped'); continue
    cell = cl[0]; t0 = time.time(); row, preds, y_true, _sc = run_one(cell, 0, 'SMOKE|' + make_run_id(cell, 0), tempfile.gettempdir(), epochs_override=2, save=False)
    assert len(preds) == len(y_true) == row['test_n'] and row['epochs_run'] <= 2 and row['integrity_pass'] is True, f'{t}: smoke run failed integrity: {row}'
    print(f"   [PASS] {t} B{cell['budget']} k{cell['k']} {row['states']}: {row['primary_view']} mF1 {_pct(row['primary_f1'])} | paper-rule mF1 {_pct(row['paper_f1'])} | integrity PASS | {time.time() - t0:.1f}s (2 epochs: skill is not meaningful)")
SMOKE_OK = True; print('\nSMOKE TEST PASSED  -> run the seed-0 cell next.')

## Run one seed (set `RUN_SEED`; resumable)

In [ ]:
# ======================================================================================
# LEVEL 1 RUN, ONE SEED PER EXECUTION (resumable).  Seed 0 first -> gate report -> your approval -> seed 1 ... seed 4 (each with its own gate report).
#   Set RUN_SEED in the config cell.  Re-running this cell resumes: finished runs are skipped.  Aborts only after 5 consecutive INTEGRITY failures.
# ======================================================================================
assert TESTS_PASSED and SMOKE_OK and APPROVED['plan'], 'test gate, smoke test and plan approval are required'
assert RUN_SEED in SEEDS, f'RUN_SEED must be one of {SEEDS}'
RESULTS_CSV = os.path.join(OUT_DIR, 'results.csv'); LOG_PATH = os.path.join(OUT_DIR, 'run_log.txt')
if RUN_SEED > 0:
    assert APPROVED[f'seed{RUN_SEED - 1}_gate_report'], f'seed {RUN_SEED - 1} gate report has not been approved (APPROVED["seed{RUN_SEED - 1}_gate_report"])'
    _prev = load_results(RESULTS_CSV); _n = int(((pd.to_numeric(_prev['seed'], errors='coerce') == RUN_SEED - 1) & _prev['status'].isin(['ok', 'integrity_fail'])).sum()) if len(_prev) else 0
    assert _n == len(PLAN_CELLS_RUN), f'seed {RUN_SEED - 1} is incomplete ({_n}/{len(PLAN_CELLS_RUN)} runs) -- finish it first'
check_namespace(write=True)
os.makedirs(os.path.join(OUT_DIR, 'runs'), exist_ok=True); os.makedirs(os.path.join(OUT_DIR, 'gate_reports'), exist_ok=True)
run_seed(RUN_SEED, PLAN_CELLS_RUN, RESULTS_CSV, OUT_DIR, LOG_PATH)
_df = load_results(RESULTS_CSV); _report = gate_report(_df, RUN_SEED, PLAN_CELLS_RUN)
atomic_write(os.path.join(OUT_DIR, 'gate_reports', f'seed{RUN_SEED}.txt'), lambda f: f.write(_report), mode='w')
print('\n' + _report)
print(f'\n>>> Gate report saved. To continue: review it, then set APPROVED["seed{RUN_SEED}_gate_report"] = True and RUN_SEED = {RUN_SEED + 1}.' if RUN_SEED < SEEDS[-1] else '\n>>> All seeds done: run the summary cell.')

## Summary, statistics and figures

In [ ]:
# ======================================================================================
# SUMMARY (safe to run at any time).  Unit = state combination; the seeds are averaged FIRST.  Binary (ARG/CHN) and multiclass (AUS/FRA) F1 are never compared with each other.
# ======================================================================================
import matplotlib
if not IN_NOTEBOOK: matplotlib.use('Agg')
import matplotlib.pyplot as plt
RESULTS_CSV = os.path.join(OUT_DIR, 'results.csv'); df = load_results(RESULTS_CSV); _check(len(df) > 0, 'no results yet')
print('runs by status:', df['status'].value_counts().to_dict()); ok = df[df['status'] == 'ok']
print(f'seeds present: {sorted(pd.to_numeric(ok.seed).unique().astype(int).tolist())} (final statistics need all of {SEEDS}); integrity-failed / errored runs are EXCLUDED from the statistics')
CS = cell_stats(df); GAIN = gain_kmin_kmax(CS); REG = regression_report(df); SEED_WARN = seed_direction_warnings(df)
os.makedirs(os.path.join(OUT_DIR, 'tables'), exist_ok=True)
for name, t in [('cell_stats', CS), ('gain_kmin_kmax', GAIN), ('regression', REG)]: atomic_write(os.path.join(OUT_DIR, 'tables', f'{name}.csv'), lambda f, t=t: t.to_csv(f, index=False), mode='w')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30)
print('\n--- per cell (primary F1 in %; k>=2: 95% t-interval with finite-population correction over combinations; k=1 census: no CI; jackknife = delete-one-state interval) ---')
show = CS.copy()
for c in ['mean', 'ci_lo', 'ci_hi', 'jackknife_lo', 'jackknife_hi', 'seed_sd_within_combo', 'seed_sd_of_cell_means']: show[c] = (100 * show[c]).round(2)
print(show.drop(columns=['census_values', 'se', 'jackknife_se']).to_string(index=False))
print('\n--- k=1 census values (every pool state, no CI) ---')
for r in CS[CS.census].itertuples(): print(f'  {r.target} B{r.budget}: {r.census_values}')
print('\n--- seed spread (reported separately from the combination interval): within-combination SD over seeds and SD of the seed-level cell means are the two columns above ---')
print('\n--- gain k_min -> k_max (percentage points) with CI ---'); g2 = GAIN.copy()
for c in ['gain', 'ci_lo', 'ci_hi']: g2[c] = (100 * g2[c]).round(2)
print(g2.to_string(index=False))
print('\n--- regression of combination-level F1 on k, key-crop count, budget (per target) ---'); print(REG.round(5).to_string(index=False))
print('   NOTE: key-crop count = key share x budget exactly, so it cannot be separated from budget; the model is fitted without it when aliased.  Combination-level rows from different budgets reuse the same states (not independent).')
print('\n--- seed-direction check (WARNING only) ---'); print('\n'.join(SEED_WARN) if SEED_WARN else '  no disagreement')
print('\n--- all-other / majority baselines vs model (mean over runs; primary view) ---')
b = ok.copy()
for c in ['primary_f1', 'baseline_all_other_primary', 'baseline_majority_primary', 'paper_f1', 'cov_cond_primary_f1']: b[c] = pd.to_numeric(b[c], errors='coerce')
print((100 * b.groupby(['target', 'primary_view'])[['primary_f1', 'baseline_all_other_primary', 'baseline_majority_primary', 'paper_f1', 'cov_cond_primary_f1']].mean()).round(2).to_string())
print('\nContext only (NOT comparable: full-USA source, augmentation on, paper Table 3 CropNet_AUGM mF1 %):', TABLE3_REF_MF1)
fl = ok['skill_flags'].fillna('').astype(str).str.split('|').explode().replace('', np.nan).dropna(); print('skill flags raised:', fl.str.replace(r':.*', '', regex=True).value_counts().to_dict())

tg = [t for t in TARGETS_ORDER if t in set(CS.target)]
if tg:
    fig, axes = plt.subplots(1, len(tg), figsize=(5.2 * len(tg), 4.2), squeeze=False)
    for ax, t in zip(axes[0], tg):
        for B, g in CS[CS.target == t].groupby('budget'):
            ax.errorbar(g.k, g['mean'], yerr=[(g['mean'] - g.ci_lo).fillna(0), (g.ci_hi - g['mean']).fillna(0)], marker='o', capsize=3, label=f'B={B}')
        bl = pd.to_numeric(ok[ok.target == t].baseline_all_other_primary).mean(); ax.axhline(bl, color='grey', ls=':', label='all-other baseline')
        ax.set_xscale('log', base=2); ax.set_xticks(sorted(CS[CS.target == t].k.unique())); ax.set_xticklabels(sorted(CS[CS.target == t].k.unique())); ax.set_xlabel('k (states)')
        ax.set_ylabel('macro-F1 (' + ok[ok.target == t].primary_view.iloc[0] + ')'); ax.set_title(f'{t} (k=1: census, no CI)'); ax.legend(fontsize=7)
    fig.tight_layout(); atomic_write(os.path.join(OUT_DIR, 'tables', 'primary_f1_vs_k.png'), lambda f: fig.savefig(f, dpi=130), mode='wb'); plt.show() if IN_NOTEBOOK else None

## Optional: export the state table for Kaggle

In [ ]:
# ======================================================================================
# OPTIONAL: export the tiny USA state table so the KAGGLE notebook can skip the raw-data download entirely
# ======================================================================================
_src = os.path.join(OUT_DIR, 'cache', 'usa_meta.csv'); _dst = os.path.join(OUT_DIR, 'for_kaggle'); os.makedirs(_dst, exist_ok=True)
import shutil; shutil.copy(_src, os.path.join(_dst, 'usa_meta.csv'))
print('Upload this file as a Kaggle Dataset and attach it to the Kaggle notebook:\n  ', os.path.join(_dst, 'usa_meta.csv'))